# SAM-CD (FastSAM-x) Bitemporal Flood-Change Detection — S1GFloods Benchmark
### Drop-in benchmark partner for the Siamese-UNet notebook: same data, split, preprocessing, augmentation, optimiser, schedule, loss, threshold search, metrics and exports

## Model chosen for the benchmark
**SAM-CD** — L. Ding, K. Zhu, D. Peng, H. Tang, K. Yang, L. Bruzzone, *"Adapting Segment Anything Model for Change Detection in VHR Remote Sensing Images"*, **IEEE Transactions on Geoscience and Remote Sensing, vol. 62, 2024**, DOI 10.1109/TGRS.2024.3368168. Official code: https://github.com/ggsDing/SAM-CD

Why it is a defensible benchmark partner:
- It is a **bitemporal, Siamese** change-detection network built on a Segment-Anything-family foundation encoder (FastSAM), published in a Q1 journal, with public code.
- It consumes a PRE/POST pair and returns a single change logit map, so it plugs into exactly the same pipeline as `siamese_unet` (`model(pre, post) -> [B,1,H,W]` logits).
- Architecture: frozen FastSAM-x encoder (multi-scale features at strides 4/8/16/32) → convolutional adaptors → shared decoder per date → concatenation → 6 residual blocks → spatial-attention-gated change head. The paper additionally proposes a *task-agnostic semantic learning* (TSL) branch/loss; it is available here as an **optional** switch (`SAMCD_USE_TSL`, default `False`) so the default loss is identical to the baseline.

## Training settings: identical to the Siamese-UNet notebook (not edited)
| Item | Value (unchanged) |
|---|---|
| Data / split | `DATA_ROOT` flat `A/B/Label`, deterministic natural-filename split 4300 / 530 / 530 |
| Preprocessing | `S1GFloodsPreprocessor`, `gray/255`, `INPUT_MODE="1ch"`, same 256×256 patches |
| Augmentation | same synced flips/rot90 + same gray-domain radiometric augmentation (PRE/POST never swapped) |
| Epochs / batch | `EPOCHS=200`, `BATCH_SIZE_PER_GPU=4`, `ACCUM_STEPS=4` (effective 16) |
| Optimiser / LR | AdamW, `LEARNING_RATE=1e-4`, `WEIGHT_DECAY=1e-4`, grad-clip 1.0 |
| Schedule | 5-epoch linear warm-up + cosine |
| Loss | `bce_dice` (0.5 / 0.5), masked to valid pixels |
| Selection / stopping | best `val_iou`, early-stopping patience 50 |
| Precision / seed | AMP on, seed 42, `RUN_SEEDS=[42]` |
| Threshold | validation-only search 0.10–0.90 (33 steps), locked, TTA off |
| Metrics / reports | same test metrics, bootstrap CIs, qualitative figures, GeoTIFF inference, parity test, manifest |
| GPUs | **single GPU** — the Siamese-UNet notebook hard-sets `EFFECTIVE_DDP=False` and `EFFECTIVE_DP=False`, so no multi-GPU is used here either |

## What necessarily differs (be explicit about these in the paper)
1. **Encoder is frozen and pretrained.** SAM-CD keeps the FastSAM-x encoder frozen (as in the official code) and trains only the adaptors/decoders/change head. The Siamese-UNet is trained from scratch. Total parameters are therefore much larger, but *trainable* parameters are small — both are reported.
2. **Gray → 3 channels inside the model.** FastSAM expects RGB in [0,1]. The data pipeline still produces the identical single-channel VV tensor (`INPUT_MODE="1ch"`); the model wrapper replicates it to 3 channels internally. No other normalisation is applied (the official SAM-CD code also feeds `image/255`).
3. **Encoder features are captured with forward hooks / a manual layer loop** instead of the official line-number patches to the Ultralytics source (which are brittle). The tapped layers are the same ones the patch exposes: layers 15, 18, 21 (neck P3/P4/P5) and layer 1 (stride-4 backbone), i.e. 320/640/640/160 channels.
4. **Domain gap.** FastSAM was pretrained on optical RGB (SA-1B). A frozen encoder on single-band SAR gray values may under-perform a from-scratch network; that is a legitimate finding of the benchmark, not a bug.
5. `MAX_TRAIN_HOURS` (default 10.5) is a safety guard so a Kaggle 12 h session still reaches evaluation/export. It never triggers if training finishes earlier.

## Before you hit *Run All* on Kaggle
- Accelerator: **GPU** (T4 / P100). **Internet: ON** (installs `ultralytics`, downloads `FastSAM-x.pt` ≈ 145 MB). If you cannot enable Internet, attach a dataset containing `FastSAM-x.pt`; the notebook finds it under `/kaggle/input`.
- `DATA_ROOT` already points to the same S1GFloods path as the baseline notebook — edit only if yours differs.
- Optional: set `BASELINE_RESULTS_DIR` (Cell 4) to the output folder of the Siamese-UNet run to get a side-by-side comparison table with a paired bootstrap on per-image IoU.

## Unchanged from the Siamese-UNet notebook
The PNG audit, label-semantics audit, `S1GFloodsPreprocessor`, augmentation classes, dataset/loader, loss zoo, metrics, checkpointing, validation-threshold search, test evaluation, bootstrap CIs, qualitative/EPS exports, complexity report, GeoTIFF inference + parity test, radiometric sensitivity test, domain-shift report, manifest and zip export.

## Verified dataset facts / UNVERIFIED assumptions
Same as the baseline notebook: `PNG_DB_LO=-30`, `PNG_DB_HI=0` is an inferred, **unverified** dB→gray mapping; `EXTERNAL_INPUT_IS_COMPOSITE=True` is an editable assumption; split independence cannot be verified because filenames carry no event/region ID.


In [1]:
import os
import shutil

WIPE_WORKING = False  # <<< EDIT THIS — destructive; keep False unless you explicitly want a clean /kaggle/working
WORKING_DIR = "/kaggle/working"

if WIPE_WORKING:
    print(f"WIPE_WORKING=True: deleting contents of {WORKING_DIR}")
    for filename in os.listdir(WORKING_DIR):
        file_path = os.path.join(WORKING_DIR, filename)
        try:
            if os.path.isfile(file_path) or os.path.islink(file_path):
                os.unlink(file_path)
            elif os.path.isdir(file_path):
                shutil.rmtree(file_path)
        except Exception as e:
            print(f"Failed to delete {file_path}. Reason: {e}")
else:
    print("WIPE_WORKING=False: preserving existing /kaggle/working contents.")

WIPE_WORKING=False: preserving existing /kaggle/working contents.


In [2]:
# Cell 1A — S1GFloods split: same deterministic natural-filename split as the Siamese-UNet notebook (Cell 8).
print("S1GFloods split source: DATA_ROOT/A, DATA_ROOT/B, DATA_ROOT/Label -> deterministic natural-filename split 4300/530/530.")
print("Identical split => the SAM-CD vs Siamese-UNet comparison is on exactly the same train/val/test images.")


S1GFloods split source: DATA_ROOT/A, DATA_ROOT/B, DATA_ROOT/Label -> deterministic natural-filename split 4300/530/530.
Identical split => the SAM-CD vs Siamese-UNet comparison is on exactly the same train/val/test images.


In [3]:
# Cell 2 — Install dependencies
# Kaggle images already ship torch/torchvision with CUDA, so we do NOT reinstall torch (that risks breaking the GPU build).
# We only add the extra geospatial / metrics / plotting packages we need.
import sys, subprocess

def pip_install(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps"] + pkgs, check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)

REQUIRED = [
    "rasterio>=1.3.9",
    "scikit-learn>=1.3.0",
    "scikit-image>=0.22.0",
    "albumentations>=1.4.0",
    "tifffile>=2023.9.26",
    "pandas>=2.0.0",
    "matplotlib>=3.7.0",
    "seaborn>=0.13.0",
    "tqdm>=4.66.0",
    "openpyxl>=3.1.0",
    "thop>=0.1.1.post2209072238",   # FLOPs/MACs estimation
    "einops>=0.7.0",                # tensor reshaping for BIT / ChangeFormer attention
]

try:
    pip_install(REQUIRED)
    print("Dependency installation finished.")
except subprocess.CalledProcessError as e:
    print("WARNING: some packages failed to install via pip. Notebook will still run; "
          "optional features (e.g. FLOPs counting) may be skipped.", e)

Dependency installation finished.


In [4]:
# Cell 2B — SAM-CD dependency: Ultralytics (used ONLY to un-pickle the FastSAM-x encoder weights).
# No Ultralytics source files are patched; multi-scale features are captured in Cell 16 by iterating the layers.
# Kaggle's preinstalled torch / numpy / opencv are left untouched whenever possible.
import subprocess, sys

ULTRALYTICS_SPEC = "ultralytics==8.3.0"   # any 8.x exposing ultralytics.nn.tasks.SegmentationModel works

def _ultralytics_importable():
    code = ("import ultralytics; from ultralytics.nn.tasks import SegmentationModel; "
            "print(ultralytics.__version__)")
    r = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True)
    return r.returncode == 0, (r.stdout.strip() if r.returncode == 0 else (r.stderr.strip()[-800:]))

ok, msg = _ultralytics_importable()
if not ok:
    print("Ultralytics not importable yet -> light install (--no-deps).")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                    ULTRALYTICS_SPEC, "ultralytics-thop", "py-cpuinfo"], check=False)
    ok, msg = _ultralytics_importable()
if not ok:
    print("Light install insufficient -> full install of", ULTRALYTICS_SPEC)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", ULTRALYTICS_SPEC], check=False)
    ok, msg = _ultralytics_importable()
if not ok:
    raise RuntimeError("Could not import ultralytics. Enable Internet in the Kaggle notebook settings "
                       "(or attach an ultralytics wheel dataset). Last error:\n" + msg)
print("ultralytics OK, version:", msg)


Ultralytics not importable yet -> light install (--no-deps).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 881.3/881.3 kB 13.0 MB/s eta 0:00:00
ultralytics OK, version: Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
8.3.0


In [5]:
# Cell 2C — Obtain the FastSAM-x checkpoint (the SAM-family encoder used by SAM-CD).
import os, glob, shutil, urllib.request

FASTSAM_DIR = "/kaggle/working/pretrained_weight"
FASTSAM_PATH = os.path.join(FASTSAM_DIR, "FastSAM-x.pt")
FASTSAM_URLS = [
    "https://github.com/ultralytics/assets/releases/download/v8.2.0/FastSAM-x.pt",
    "https://github.com/ultralytics/assets/releases/latest/download/FastSAM-x.pt",
]
FASTSAM_MIN_BYTES = 100 * 1024 * 1024    # the official file is ~145 MB; anything smaller is a failed download
os.makedirs(FASTSAM_DIR, exist_ok=True)

def _fastsam_ok(p):
    return os.path.isfile(p) and os.path.getsize(p) >= FASTSAM_MIN_BYTES

if _fastsam_ok(FASTSAM_PATH):
    print("FastSAM-x.pt already present:", FASTSAM_PATH)
else:
    # 1) a copy attached as a Kaggle dataset (works with Internet OFF)
    local = []
    for pat in ["/kaggle/input/*/FastSAM-x.pt", "/kaggle/input/*/*/FastSAM-x.pt", "/kaggle/input/*/*/*/FastSAM-x.pt"]:
        local += glob.glob(pat)
    local = [p for p in local if _fastsam_ok(p)]
    if local:
        shutil.copy(local[0], FASTSAM_PATH)
        print("Copied FastSAM-x.pt from attached input:", local[0])
    else:
        # 2) download
        last_err = None
        for url in FASTSAM_URLS:
            try:
                print("Downloading", url)
                req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
                with urllib.request.urlopen(req, timeout=120) as r, open(FASTSAM_PATH + ".part", "wb") as f:
                    shutil.copyfileobj(r, f, length=1 << 20)
                os.replace(FASTSAM_PATH + ".part", FASTSAM_PATH)
                if _fastsam_ok(FASTSAM_PATH):
                    break
                last_err = "downloaded file too small"
            except Exception as e:
                last_err = repr(e)
                print("  failed:", last_err)
        if not _fastsam_ok(FASTSAM_PATH):
            raise RuntimeError("Could not obtain FastSAM-x.pt (" + str(last_err) + "). Turn Internet ON in the Kaggle "
                               "settings, or attach a dataset that contains FastSAM-x.pt.")
print(f"FastSAM-x.pt ready: {FASTSAM_PATH} ({os.path.getsize(FASTSAM_PATH)/1e6:.1f} MB)")


FastSAM-x.pt ready: /kaggle/working/pretrained_weight/FastSAM-x.pt (145.0 MB)


In [6]:
# Cell 3 — Imports
import os, re, io, json, glob, math, time, random, shutil, zipfile, warnings, platform, subprocess, getpass
import hashlib
import contextlib
from pathlib import Path
from dataclasses import dataclass, field, asdict
from collections import defaultdict, OrderedDict
from typing import Optional, List, Tuple, Dict, Any

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from scipy.ndimage import gaussian_filter, gaussian_filter1d, label as cc_label
from scipy.stats import wasserstein_distance
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
import torch.multiprocessing as mp
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as TorchDDP
from torch.utils.data.distributed import DistributedSampler

import einops
from einops import rearrange

import rasterio
from rasterio.windows import Window
from rasterio.transform import Affine, from_origin

from sklearn.metrics import (
    roc_auc_score, average_precision_score, cohen_kappa_score,
    matthews_corrcoef, balanced_accuracy_score, confusion_matrix
)

warnings.filterwarnings("ignore", category=rasterio.errors.NotGeoreferencedWarning)
sns.set_theme(style="whitegrid")
print("Imports OK.")

Imports OK.


In [7]:
# Cell 4 — Central configuration. Edit paths/settings marked "<<< EDIT THIS".
# Every downstream cell reads from CONFIG / these constants; do not hardcode experiment settings elsewhere.

# ---- Paths --------------------------------------------------------------
DATA_ROOT = "/kaggle/input/datasets/ashrif411/s1gfloods"  # <<< EDIT THIS
OUTPUT_ROOT = "/kaggle/working/samcd_s1gfloods"

EXTERNAL_PRE_TIF = "/kaggle/input/datasets/ashrif411/gaibandha/SPARRSO/Gaibandha_S1_pre_June 8_20 2024.tif"  # <<< EDIT THIS
EXTERNAL_POST_TIF = "/kaggle/input/datasets/ashrif411/gaibandha/SPARRSO/Gaibandha_S1_post_June 25_July 10_2024.tif"  # <<< EDIT THIS
EXTERNAL_REF_MASK = None  # <<< EDIT THIS — optional reference-mask GeoTIFF; keep None if unavailable
EXTERNAL_REF_POSITIVE_VALUE = None  # <<< EDIT THIS — None means any valid value > 0 is positive

# ---- Dataset / input domain --------------------------------------------
INPUT_MODE = "1ch"  # <<< EDIT THIS — "1ch" | "3ch_replicated"
if INPUT_MODE not in {"1ch", "3ch_replicated"}:
    raise ValueError("INPUT_MODE must be '1ch' or '3ch_replicated'.")
IN_CHANNELS = 1 if INPUT_MODE == "1ch" else 3

# UNVERIFIED: inferred from measured S1GFloods gray statistics; not confirmed by the dataset authors.
PNG_DB_LO = -30.0  # <<< EDIT THIS
PNG_DB_HI = 0.0    # <<< EDIT THIS
PNG_QUANTIZE = True
MASK_BOTH_ZERO = False  # <<< EDIT THIS — enable only if the zero-border audit supports footprint NoData masking

# ---- Experiment identity ----------------------------------------------
MODEL_NAME = "sam_cd"  # sam_cd | bitemporal_unet | siamese_unet | fc_siam_conc | fc_siam_diff | attention_unet | bit  (benchmark run: sam_cd)
TARGET_MODE = "newly_inundated_flood"  # S1GFloods GT semantics supplied by the user

# ---- SAM-CD (Ding et al., IEEE TGRS 2024) -----------------------------
# Encoder = frozen FastSAM-x (as in the official code). Everything else in this cell is identical to the baseline.
SAMCD_WEIGHTS = FASTSAM_PATH          # set by Cell 2C
SAMCD_NUM_EMBED = 8                   # official default: channels of the semantic-latent heads
SAMCD_USE_TSL = False                 # <<< EDIT THIS — paper's optional task-agnostic semantic loss. False => loss identical to the baseline (bce_dice only)
SAMCD_TSL_TEMPERATURE = 3.0           # official LatentSimilarity(T=3.0); only used if SAMCD_USE_TSL=True
SAMCD_TSL_WEIGHT = 1.0                # only used if SAMCD_USE_TSL=True
MAX_TRAIN_HOURS = 10.5                # <<< EDIT THIS — safety guard for Kaggle's 12 h session; None disables it

# Optional: output folder of the finished Siamese-UNet run (contains metrics/test_metrics_per_image.csv) for the comparison cell.
BASELINE_RESULTS_DIR = None           # <<< EDIT THIS — e.g. "/kaggle/input/siamese-unet-run/changeMamba_s1gfloods"

# ---- Training hyperparameters -----------------------------------------
# Reference defaults requested for S1GFloods: 256 patch, batch ~16 effective, lr=1e-4, 100 epochs, hybrid loss.
EPOCHS = 200
BATCH_SIZE_PER_GPU = 4  # kept identical to the Siamese-UNet run (x ACCUM_STEPS=4 -> effective 16); do not change for the benchmark
ACCUM_STEPS = 4         # <<< EDIT THIS — 4×4=16 on one GPU; use 2 with two-GPU DDP for global effective batch 16
NUM_WORKERS = 0

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0
SCHEDULER_NAME = "cosine"
WARMUP_EPOCHS = 5

LOSS_NAME = "bce_dice"  # bce | dice | bce_dice | focal | focal_dice | tversky | bce_tversky | focal_tversky
BCE_WEIGHT = 0.5
DICE_WEIGHT = 0.5
FOCAL_GAMMA = 2.0
FOCAL_ALPHA = 0.25
TVERSKY_ALPHA = 0.3
TVERSKY_BETA = 0.7
EARLY_STOPPING_PATIENCE = 50
CHECKPOINT_METRIC = "val_iou"

SEED = 42
RUN_SEEDS = [42]  # <<< EDIT THIS to [42, 52, 62, 72, 82] for final multi-seed reporting
USE_AMP = True
USE_DDP = True
USE_AUGMENTATION = True
USE_SAR_RADIOMETRIC_AUG = True

# Gray-domain radiometric robustness controls (training only).
AUG_DARK_TAIL = True
AUG_GAIN_OFFSET = True
AUG_DATE_OFFSET = True
AUG_BLUR = True

# ---- BIT ---------------------------------------------------------------
BIT_DIM = 32
BIT_TOKENS = 4
BIT_ENC_DEPTH = 1
BIT_DEC_DEPTH = 8
BIT_HEADS = 8

# ---- Inference ---------------------------------------------------------
INFERENCE_TILE_SIZE = 256
INFERENCE_OVERLAP = 128
INFERENCE_TTA = False
AUTO_ALIGN = False
NODATA_FILL = "median"  # <<< EDIT THIS — "median" | "zero"
EXTERNAL_INPUT_IS_COMPOSITE = True  # <<< EDIT THIS — UNVERIFIED assumption written to metadata

# Optional external two-point calibration; OFF by default and approximate when used.
EXTERNAL_CALIBRATION = "none"  # <<< EDIT THIS — "none" | "two_point"
CALIBRATION_STABLE_LAND_DB = None  # <<< EDIT THIS if EXTERNAL_CALIBRATION="two_point"
CALIBRATION_RIVER_DB = None        # <<< EDIT THIS if EXTERNAL_CALIBRATION="two_point"
CALIBRATION_TARGET_LAND_GRAY = 144.0
CALIBRATION_TARGET_WATER_GRAY = 47.0

# User-selected external ROI for the radiometric sensitivity test: (col_off, row_off, width, height).
ROI_WINDOW = None  # <<< EDIT THIS — e.g. (1000, 1500, 1024, 1024)
SENSITIVITY_DB_RANGES = [(-30.0, 0.0), (-28.0, -2.0), (-32.0, -2.0), (-30.0, -2.0)]
SENSITIVITY_NODATA_FILLS = ["median", "zero"]
PARITY_TEST_PAIRS = 3
PARITY_TOLERANCE = 1e-2
DOMAIN_FIGURE_MAX_SIZE = 1024

# ---- Threshold ---------------------------------------------------------
THRESHOLD = 0.5
THRESHOLD_SEARCH_RANGE = (0.10, 0.90)
THRESHOLD_SEARCH_STEPS = 33

# ---- Resume / reporting ------------------------------------------------
RESUME_CHECKPOINT = None  # <<< EDIT THIS
BOOTSTRAP_REPS = 1000
BOOTSTRAP_CI = 0.95

print("UNVERIFIED: PNG dB→gray mapping assumed linear over", (PNG_DB_LO, PNG_DB_HI))
print("UNVERIFIED: EXTERNAL_INPUT_IS_COMPOSITE =", EXTERNAL_INPUT_IS_COMPOSITE)
print("Input mode:", INPUT_MODE, "| IN_CHANNELS:", IN_CHANNELS)


def _is_json_serializable(v):
    try:
        json.dumps(v)
        return True
    except (TypeError, ValueError):
        return False

CONFIG: Dict[str, Any] = {
    k: v for k, v in dict(globals()).items()
    if k.isupper() and not k.startswith("_") and k not in ("CONFIG",)
    and _is_json_serializable(v)
}
print(f"Loaded {len(CONFIG)} config keys.")

UNVERIFIED: PNG dB→gray mapping assumed linear over (-30.0, 0.0)
UNVERIFIED: EXTERNAL_INPUT_IS_COMPOSITE = True
Input mode: 1ch | IN_CHANNELS: 1
Loaded 85 config keys.


In [8]:
# Cell 5 — Reproducibility
def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    # Deterministic where practical. Full determinism on cuDNN convs costs throughput; we trade a little
    # determinism for speed on the (already algorithmically-controlled) conv backend but fix everything else.
    torch.backends.cudnn.benchmark = True
    torch.use_deterministic_algorithms(False)  # True is safer for bit-exactness but breaks some cuDNN kernels/ops we use
    return seed

set_global_seed(SEED)
print(f"Global seed set to {SEED}")


Global seed set to 42


In [9]:
# Cell 6 — GPU / environment info
def print_environment_info():
    print("="*70)
    print("Python       :", platform.python_version())
    print("PyTorch      :", torch.__version__)
    print("CUDA (torch) :", torch.version.cuda)
    print("cuDNN        :", torch.backends.cudnn.version())
    print("CUDA available:", torch.cuda.is_available())
    n_gpu = torch.cuda.device_count()
    print("GPU count    :", n_gpu)
    for i in range(n_gpu):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name}, {p.total_memory/1e9:.1f} GB, SM {p.major}.{p.minor}")
    print("="*70)
    return n_gpu

N_GPUS = print_environment_info()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#EFFECTIVE_DDP = USE_DDP and N_GPUS > 1
EFFECTIVE_DDP = False
#EFFECTIVE_DP = (not EFFECTIVE_DDP) and N_GPUS > 1
EFFECTIVE_DP = False
print(f"Strategy -> DDP: {EFFECTIVE_DDP} | DataParallel: {EFFECTIVE_DP} | single-device: {not (EFFECTIVE_DDP or EFFECTIVE_DP)}")

print("Strategy mirrors the Siamese-UNet notebook: single GPU, DDP/DataParallel disabled (like-for-like benchmark).")


Python       : 3.13.15
PyTorch      : 2.11.0+cu128
CUDA (torch) : 12.8
cuDNN        : 91900
CUDA available: True
GPU count    : 2
  GPU 0: Tesla T4, 15.6 GB, SM 7.5
  GPU 1: Tesla T4, 15.6 GB, SM 7.5
Strategy -> DDP: False | DataParallel: False | single-device: True
Strategy mirrors the Siamese-UNet notebook: single GPU, DDP/DataParallel disabled (like-for-like benchmark).


In [10]:
# Cell 7 — Output directory structure
SUBDIRS = ["checkpoints", "logs", "metrics", "figures", "predictions", "qualitative",
           "configs", "inference_outputs", "environment", "split_audit"]

def make_output_dirs(root: str) -> Dict[str, str]:
    paths = {}
    os.makedirs(root, exist_ok=True)
    for d in SUBDIRS:
        p = os.path.join(root, d)
        os.makedirs(p, exist_ok=True)
        paths[d] = p
    return paths

DIRS = make_output_dirs(OUTPUT_ROOT)

with open(os.path.join(DIRS["configs"], "config.json"), "w") as f:
    json.dump(CONFIG, f, indent=2, default=str)

print("Output root:", OUTPUT_ROOT)
for k, v in DIRS.items():
    print(f"  {k:<20s} -> {v}")


Output root: /kaggle/working/samcd_s1gfloods
  checkpoints          -> /kaggle/working/samcd_s1gfloods/checkpoints
  logs                 -> /kaggle/working/samcd_s1gfloods/logs
  metrics              -> /kaggle/working/samcd_s1gfloods/metrics
  figures              -> /kaggle/working/samcd_s1gfloods/figures
  predictions          -> /kaggle/working/samcd_s1gfloods/predictions
  qualitative          -> /kaggle/working/samcd_s1gfloods/qualitative
  configs              -> /kaggle/working/samcd_s1gfloods/configs
  inference_outputs    -> /kaggle/working/samcd_s1gfloods/inference_outputs
  environment          -> /kaggle/working/samcd_s1gfloods/environment
  split_audit          -> /kaggle/working/samcd_s1gfloods/split_audit


In [11]:
# Cell 8 — S1GFloods flat-folder discovery + deterministic split
# Dataset layout:
# DATA_ROOT/A/
# DATA_ROOT/B/
# DATA_ROOT/Label/
#
# IMPORTANT:
# No official split metadata is available in this dataset copy.
# Therefore a deterministic natural-filename split is constructed here.
# Label/ is treated logically as GT/; files are NOT renamed or copied.

from pathlib import Path
import re
import pandas as pd

EXPECTED_COUNTS = {
    "train": 4300,
    "val": 530,
    "test": 530,
}

VALID_SPLITS = ("train", "val", "test")


def natural_key(name):
    """
    Natural filename ordering:
    image_2.png comes before image_10.png.
    """
    return [
        int(x) if x.isdigit() else x.lower()
        for x in re.split(r"(\d+)", str(name))
    ]


def find_role_folder(root, possible_names):
    """
    Case-insensitive folder lookup.
    For example Label, label, LABEL are all accepted.
    """
    root = Path(root)

    dirs = {
        p.name.lower(): p
        for p in root.iterdir()
        if p.is_dir()
    }

    for name in possible_names:
        if name.lower() in dirs:
            return dirs[name.lower()]

    raise FileNotFoundError(
        f"Could not find any of {possible_names} under {root}. "
        f"Found folders: {[p.name for p in root.iterdir() if p.is_dir()]}"
    )


def png_map(folder):
    """Map filename -> full path."""
    files = list(folder.glob("*.png")) + list(folder.glob("*.PNG"))

    mapping = {}
    for p in files:
        if p.name in mapping:
            raise RuntimeError(f"Duplicate filename detected: {p.name}")
        mapping[p.name] = str(p)

    return mapping


def discover_and_split_s1gfloods(root):
    root = Path(root)

    if not root.exists():
        raise FileNotFoundError(f"DATA_ROOT does not exist: {root}")

    # ---------------------------------------------------------
    # Locate A / B / Label
    # Label is treated as GT logically; nothing is renamed.
    # ---------------------------------------------------------
    a_dir = find_role_folder(root, ["A"])
    b_dir = find_role_folder(root, ["B"])
    label_dir = find_role_folder(root, ["Label", "GT"])

    print("Detected folders:")
    print("  PRE   :", a_dir)
    print("  POST  :", b_dir)
    print("  LABEL :", label_dir)
    print("  NOTE  : Label/ is treated as GT; files are not renamed.")

    # ---------------------------------------------------------
    # Collect filenames
    # ---------------------------------------------------------
    a_files = png_map(a_dir)
    b_files = png_map(b_dir)
    label_files = png_map(label_dir)

    a_names = set(a_files)
    b_names = set(b_files)
    l_names = set(label_files)

    # Strict pairing check
    if not (a_names == b_names == l_names):
        print("\nFilename mismatch detected:")
        print("A only:", sorted(a_names - b_names - l_names)[:10])
        print("B only:", sorted(b_names - a_names - l_names)[:10])
        print("Label only:", sorted(l_names - a_names - b_names)[:10])

        raise RuntimeError(
            "A, B and Label filenames do not match exactly. "
            "Fix the dataset before training."
        )

    # Natural deterministic ordering
    names = sorted(a_names, key=natural_key)

    n_total = len(names)

    print(f"\nTotal matched A/B/Label pairs: {n_total}")

    # ---------------------------------------------------------
    # Preferred split: 4300 / 530 / 530
    # ---------------------------------------------------------
    expected_total = sum(EXPECTED_COUNTS.values())  # 5360

    if n_total == expected_total:
        n_train = EXPECTED_COUNTS["train"]
        n_val = EXPECTED_COUNTS["val"]

    else:
        # Fallback using approximately the same proportions
        # 4300/5360 ≈ 80.22%
        # 530/5360  ≈ 9.89%
        print(
            f"WARNING: expected {expected_total} images but found {n_total}.\n"
            "Using proportional deterministic split instead."
        )

        train_ratio = 4300 / 5360
        val_ratio = 530 / 5360

        n_train = int(round(n_total * train_ratio))
        n_val = int(round(n_total * val_ratio))

    train_names = names[:n_train]
    val_names = names[n_train:n_train + n_val]
    test_names = names[n_train + n_val:]

    split_names = {
        "train": train_names,
        "val": val_names,
        "test": test_names,
    }

    # ---------------------------------------------------------
    # Construct EXACT structure expected by rest of notebook
    # ---------------------------------------------------------
    splits = {}

    summary_rows = []

    for split in VALID_SPLITS:

        split_list = []

        for name in split_names[split]:

            split_list.append({
                "pre": a_files[name],
                "post": b_files[name],
                "label": label_files[name],
                "key": f"{split}/{name}",
            })

        splits[split] = split_list

        summary_rows.append({
            "split": split,
            "count": len(split_list),
        })

    summary_df = pd.DataFrame(summary_rows)

    # ---------------------------------------------------------
    # Safety checks
    # ---------------------------------------------------------
    train_set = set(train_names)
    val_set = set(val_names)
    test_set = set(test_names)

    assert train_set.isdisjoint(val_set)
    assert train_set.isdisjoint(test_set)
    assert val_set.isdisjoint(test_set)

    assert (
        len(train_set) +
        len(val_set) +
        len(test_set)
        == n_total
    )

    print("\nConstructed deterministic split:")
    print(summary_df.to_string(index=False))

    print("\nFirst/last filenames:")
    for split in VALID_SPLITS:
        ns = split_names[split]
        if ns:
            print(
                f"{split:5s}: "
                f"{ns[0]}  ->  {ns[-1]}"
            )

    print(
        "\nIMPORTANT CAVEAT:\n"
        "No official split metadata or event/region identifiers were available "
        "in this dataset copy. Train/validation/test sets were therefore "
        "constructed deterministically from natural filename order. "
        "Event-level or geographic independence cannot be verified."
    )

    return splits, summary_df


# =============================================================
# Build SPLITS used by all downstream notebook cells
# =============================================================

SPLITS, split_summary = discover_and_split_s1gfloods(DATA_ROOT)

split_source = (
    "Deterministic natural-filename split from flat "
    "A/B/Label folders; no official split metadata available"
)

triplet_by_key = {
    t["key"]: t
    for split in VALID_SPLITS
    for t in SPLITS[split]
}

# Final confirmation
print("\nFinal SPLITS:")
for split in VALID_SPLITS:
    print(f"{split}: {len(SPLITS[split])}")

Detected folders:
  PRE   : /kaggle/input/datasets/ashrif411/s1gfloods/A
  POST  : /kaggle/input/datasets/ashrif411/s1gfloods/B
  LABEL : /kaggle/input/datasets/ashrif411/s1gfloods/Label
  NOTE  : Label/ is treated as GT; files are not renamed.

Total matched A/B/Label pairs: 5360

Constructed deterministic split:
split  count
train   4300
  val    530
 test    530

First/last filenames:
train: image_1.png  ->  image_4300.png
val  : image_4301.png  ->  image_4830.png
test : image_4831.png  ->  image_5360.png

IMPORTANT CAVEAT:
No official split metadata or event/region identifiers were available in this dataset copy. Train/validation/test sets were therefore constructed deterministically from natural filename order. Event-level or geographic independence cannot be verified.

Final SPLITS:
train: 4300
val: 530
test: 530


In [12]:
# Cell 9 — Split verification / audit export
# S1GFloods ships train/val/test folders directly; there is no Sen1Floods11 split manifest or Bolivia reconstruction.

split_summary.to_csv(os.path.join(DIRS["split_audit"], "split_summary.csv"), index=False)
print(split_summary.to_string(index=False))
print("Split source:", split_source)
print(
    "Split method: deterministic natural-filename ordering. "
    "No random split is used."
)

split  count
train   4300
  val    530
 test    530
Split source: Deterministic natural-filename split from flat A/B/Label folders; no official split metadata available
Split method: deterministic natural-filename ordering. No random split is used.


In [13]:
# Cell 10 — S1GFloods PNG audit
# Reports the verified/required gray-domain statistics and saves CSVs under split_audit/.


def read_png_array(path: str) -> np.ndarray:
    with Image.open(path) as im:
        return np.asarray(im)


def read_png_gray(path: str) -> np.ndarray:
    arr = read_png_array(path)
    if arr.ndim == 2:
        return arr.astype(np.uint8)
    if arr.ndim == 3 and arr.shape[2] >= 1:
        return arr[..., 0].astype(np.uint8)
    raise ValueError(f"Unsupported PNG shape {arr.shape} for {path}")


def channels_identical(arr: np.ndarray) -> bool:
    if arr.ndim == 2:
        return True
    if arr.ndim != 3 or arr.shape[2] < 3:
        return False
    return bool(np.array_equal(arr[..., 0], arr[..., 1]) and np.array_equal(arr[..., 0], arr[..., 2]))


def _longest_true_run(v: np.ndarray) -> int:
    v = np.asarray(v, dtype=np.uint8)
    if not v.any():
        return 0
    padded = np.pad(v, (1, 1), constant_values=0)
    d = np.diff(padded.astype(np.int16))
    starts = np.where(d == 1)[0]
    ends = np.where(d == -1)[0]
    return int(np.max(ends - starts)) if len(starts) else 0


def both_zero_border_audit(a: np.ndarray, b: np.ndarray) -> Dict[str, Any]:
    z = (a == 0) & (b == 0)
    if not z.any():
        return {"both_zero_fraction": 0.0, "touches_border": False, "straight_edge_like": False,
                "interior_blob_like": False, "n_zero_components": 0}
    labeled, ncomp = cc_label(z.astype(np.uint8))
    straight = False
    interior_blob = False
    for cid in range(1, ncomp + 1):
        comp = labeled == cid
        touches = bool(comp[0, :].any() or comp[-1, :].any() or comp[:, 0].any() or comp[:, -1].any())
        if touches:
            runs = [_longest_true_run(comp[0, :]), _longest_true_run(comp[-1, :]),
                    _longest_true_run(comp[:, 0]), _longest_true_run(comp[:, -1])]
            if max(runs) >= max(16, int(0.06 * max(comp.shape))):
                straight = True
        elif int(comp.sum()) >= 4:
            interior_blob = True
    touches_border = bool(z[0, :].any() or z[-1, :].any() or z[:, 0].any() or z[:, -1].any())
    return {"both_zero_fraction": float(z.mean()), "touches_border": touches_border,
            "straight_edge_like": straight, "interior_blob_like": interior_blob,
            "n_zero_components": int(ncomp)}


def hist_median(hist: np.ndarray) -> float:
    total = hist.sum()
    if total == 0:
        return float("nan")
    return float(np.searchsorted(np.cumsum(hist), (total + 1) / 2))


def density_crossover(flood_hist: np.ndarray, land_hist: np.ndarray) -> float:
    if flood_hist.sum() == 0 or land_hist.sum() == 0:
        return float("nan")
    pf = gaussian_filter1d(flood_hist.astype(np.float64) / flood_hist.sum(), sigma=1.5)
    pl = gaussian_filter1d(land_hist.astype(np.float64) / land_hist.sum(), sigma=1.5)
    diff = pf - pl
    crossings = np.where(np.signbit(diff[:-1]) != np.signbit(diff[1:]))[0]
    if len(crossings):
        scores = (pf[crossings] + pl[crossings])
        return float(crossings[int(np.argmax(scores))] + 0.5)
    support = (pf + pl) > 1e-5
    idxs = np.where(support)[0]
    return float(idxs[np.argmin(np.abs(diff[idxs]))]) if len(idxs) else float("nan")

summary_rows = []
train_patch_rows = []
nodata_rows = []
hist_A = np.zeros(256, dtype=np.int64)
hist_B = np.zeros(256, dtype=np.int64)
hist_A_flood = np.zeros(256, dtype=np.int64)
hist_A_land = np.zeros(256, dtype=np.int64)
hist_B_flood = np.zeros(256, dtype=np.int64)
hist_B_land = np.zeros(256, dtype=np.int64)
train_pos = train_total = 0

for split in VALID_SPLITS:
    for t in tqdm(SPLITS[split], desc=f"Auditing {split} PNGs"):
        a_raw = read_png_array(t["pre"])
        b_raw = read_png_array(t["post"])
        gt_raw = read_png_array(t["label"])
        a = a_raw[..., 0] if a_raw.ndim == 3 else a_raw
        b = b_raw[..., 0] if b_raw.ndim == 3 else b_raw
        gt = gt_raw[..., 0] if gt_raw.ndim == 3 else gt_raw
        a = a.astype(np.uint8); b = b.astype(np.uint8); gt = gt.astype(np.uint8)

        shape_match = a.shape == b.shape == gt.shape
        if not shape_match:
            raise ValueError(f"Shape mismatch for {t['key']}: A={a.shape}, B={b.shape}, GT={gt.shape}")
        gt_values = np.unique(gt)
        if not set(gt_values.tolist()).issubset({0, 255}):
            raise ValueError(f"Unexpected GT values for {t['key']}: {gt_values.tolist()} (expected only 0/255)")

        za = float(np.mean(a == 0)); z255a = float(np.mean(a == 255))
        zb = float(np.mean(b == 0)); z255b = float(np.mean(b == 255))
        zero_audit = both_zero_border_audit(a, b)
        nodata_rows.append({"split": split, "key": t["key"], **zero_audit})
        summary_rows.append({
            "split": split, "key": t["key"], "shape": str(a.shape),
            "a_channels_identical": channels_identical(a_raw),
            "b_channels_identical": channels_identical(b_raw),
            "a_frac_0": za, "a_frac_255": z255a, "b_frac_0": zb, "b_frac_255": z255b,
            "gt_positive_fraction": float(np.mean(gt > 127)),
        })

        if split == "train":
            flood = gt > 127
            land = ~flood
            hist_A += np.bincount(a.ravel(), minlength=256)
            hist_B += np.bincount(b.ravel(), minlength=256)
            hist_A_flood += np.bincount(a[flood], minlength=256) if flood.any() else 0
            hist_A_land += np.bincount(a[land], minlength=256) if land.any() else 0
            hist_B_flood += np.bincount(b[flood], minlength=256) if flood.any() else 0
            hist_B_land += np.bincount(b[land], minlength=256) if land.any() else 0
            train_pos += int(flood.sum()); train_total += int(gt.size)
            land_delta = (b.astype(np.float32) - a.astype(np.float32))[land]
            train_patch_rows.append({
                "key": t["key"],
                "a_patch_median": float(np.median(a)),
                "b_patch_median": float(np.median(b)),
                "land_b_minus_a_median": float(np.median(land_delta)) if land_delta.size else np.nan,
                "gt_positive_fraction": float(flood.mean()),
            })

summary_df = pd.DataFrame(summary_rows)
train_patch_df = pd.DataFrame(train_patch_rows)
nodata_df = pd.DataFrame(nodata_rows)
summary_df.to_csv(os.path.join(DIRS["split_audit"], "png_dataset_summary.csv"), index=False)
train_patch_df.to_csv(os.path.join(DIRS["split_audit"], "train_patch_statistics.csv"), index=False)
nodata_df.to_csv(os.path.join(DIRS["split_audit"], "both_zero_border_audit.csv"), index=False)

TRAIN_POSITIVE_FRACTION = train_pos / max(train_total, 1)
TRAIN_GRAY_HISTS = {"A": hist_A, "B": hist_B}
TRAIN_CLASS_HISTS = {"A_flood": hist_A_flood, "A_land": hist_A_land, "B_flood": hist_B_flood, "B_land": hist_B_land}
TRAIN_GRAY_MEDIANS = {
    "A_flood": hist_median(hist_A_flood), "A_land": hist_median(hist_A_land),
    "B_flood": hist_median(hist_B_flood), "B_land": hist_median(hist_B_land),
}
FLOOD_NONFLOOD_CROSSOVER_GRAY = density_crossover(hist_B_flood, hist_B_land)

hist_df = pd.DataFrame({
    "gray": np.arange(256), "A_all": hist_A, "B_all": hist_B,
    "A_flood": hist_A_flood, "A_land": hist_A_land,
    "B_flood": hist_B_flood, "B_land": hist_B_land,
})
hist_df.to_csv(os.path.join(DIRS["split_audit"], "train_gray_histograms.csv"), index=False)

print("A/B channel identity fractions:",
      summary_df[["a_channels_identical", "b_channels_identical"]].mean().to_dict())
print("Train A: frac gray==0 / 255 =",
      float(hist_A[0] / hist_A.sum()), float(hist_A[255] / hist_A.sum()))
print("Train B: frac gray==0 / 255 =",
      float(hist_B[0] / hist_B.sum()), float(hist_B[255] / hist_B.sum()))
print(f"Train GT positive fraction: {TRAIN_POSITIVE_FRACTION:.4%}")
print("Class-conditional gray medians:", TRAIN_GRAY_MEDIANS)
print("Per-patch land (B-A) median distribution:")
print(train_patch_df["land_b_minus_a_median"].describe(percentiles=[0.25, 0.5, 0.75]).to_string())
print(f"Flood/non-flood B-gray density crossover ≈ {FLOOD_NONFLOOD_CROSSOVER_GRAY:.1f}")
print("Patch-level A/B median distribution:")
print(train_patch_df[["a_patch_median", "b_patch_median"]].describe().to_string())
print("Both-zero NoData-border heuristic fractions:", {
    "straight_edge_like": float(nodata_df["straight_edge_like"].mean()),
    "interior_blob_like": float(nodata_df["interior_blob_like"].mean()),
    "touches_border": float(nodata_df["touches_border"].mean()),
})
print("NOTE: the border/straight-edge classification is a heuristic audit, not a verified S1GFloods NoData annotation.")
if TRAIN_POSITIVE_FRACTION < 0.10:
    print("RECOMMENDATION ONLY: positive pixels are <10%; consider LOSS_NAME='focal_tversky' or 'bce_tversky'. "
          "The notebook does NOT change LOSS_NAME automatically.")

# Example A/B/GT panel
examples = SPLITS["train"][:4]
fig, axes = plt.subplots(len(examples), 3, figsize=(12, 4 * len(examples)))
if len(examples) == 1:
    axes = axes[None, :]
for i, t in enumerate(examples):
    a = read_png_gray(t["pre"]); b = read_png_gray(t["post"]); gt = read_png_gray(t["label"])
    for ax, img, title in zip(axes[i], [a, b, gt], ["A / PRE (VV gray)", "B / POST (VV gray)", "GT (new inundation)"]):
        ax.imshow(img, cmap="gray"); ax.set_title(f"{t['key']}\n{title}" if title.startswith("A") else title); ax.axis("off")
fig.tight_layout()
fig.savefig(os.path.join(DIRS["figures"], "s1gfloods_abgt_examples.png"), dpi=200, bbox_inches="tight")
plt.close(fig)

Auditing train PNGs:   0%|          | 0/4300 [00:00<?, ?it/s]

Auditing val PNGs:   0%|          | 0/530 [00:00<?, ?it/s]

Auditing test PNGs:   0%|          | 0/530 [00:00<?, ?it/s]

A/B channel identity fractions: {'a_channels_identical': 1.0, 'b_channels_identical': 1.0}
Train A: frac gray==0 / 255 = 0.06602232112440952 0.02643187766851381
Train B: frac gray==0 / 255 = 0.09278441318245821 0.018248862333075946
Train GT positive fraction: 33.9074%
Class-conditional gray medians: {'A_flood': 136.0, 'A_land': 144.0, 'B_flood': 39.0, 'B_land': 173.0}
Per-patch land (B-A) median distribution:
count    4300.000000
mean       22.892791
std        21.443606
min       -70.000000
25%         8.000000
50%        24.000000
75%        36.000000
max       113.000000
Flood/non-flood B-gray density crossover ≈ 98.5
Patch-level A/B median distribution:
       a_patch_median  b_patch_median
count     4300.000000     4300.000000
mean       133.397674      118.134186
std         38.113132       65.370659
min          0.000000        0.000000
25%        120.000000       63.000000
50%        136.000000      137.000000
75%        160.000000      176.000000
max        240.000000      237

In [26]:
# Cell 11 — S1GFloods label-semantics audit / documentation
# VERIFIED for this notebook: GT=255 is newly inundated flood; permanent water is negative (0).
# Therefore PRE/POST order is semantically asymmetric and must NEVER be swapped as augmentation.

semantics_rows = [{
    "target": "GT > 127",
    "positive_semantics": "newly inundated flood",
    "permanent_water": "negative / GT=0",
    "temporal_order": "A=PRE, B=POST; asymmetric; no date swapping",
    "train_positive_fraction": TRAIN_POSITIVE_FRACTION,
    "median_A_flood": TRAIN_GRAY_MEDIANS["A_flood"],
    "median_A_land": TRAIN_GRAY_MEDIANS["A_land"],
    "median_B_flood": TRAIN_GRAY_MEDIANS["B_flood"],
    "median_B_land": TRAIN_GRAY_MEDIANS["B_land"],
    "density_crossover_B_gray": FLOOD_NONFLOOD_CROSSOVER_GRAY,
}]
semantics_df = pd.DataFrame(semantics_rows)
semantics_df.to_csv(os.path.join(DIRS["split_audit"], "label_semantics_audit.csv"), index=False)
print(semantics_df.to_string(index=False))

examples = SPLITS["train"][:4]
fig, axes = plt.subplots(len(examples), 4, figsize=(16, 4 * len(examples)))
if len(examples) == 1:
    axes = axes[None, :]
for i, t in enumerate(examples):
    a = read_png_gray(t["pre"]); b = read_png_gray(t["post"]); gt = read_png_gray(t["label"])
    diff = b.astype(np.float32) - a.astype(np.float32)
    panels = [(a, "A / PRE"), (b, "B / POST"), (diff, "B-A gray difference"), (gt > 127, "GT: new inundation")]
    for ax, (img, title) in zip(axes[i], panels):
        ax.imshow(img, cmap="gray" if "difference" not in title else "RdBu", vmin=(-80 if "difference" in title else None), vmax=(80 if "difference" in title else None))
        ax.set_title(f"{t['key']}\n{title}" if title == "A / PRE" else title); ax.axis("off")
fig.tight_layout()
fig.savefig(os.path.join(DIRS["figures"], "s1gfloods_label_semantics_examples.png"), dpi=200, bbox_inches="tight")
plt.close(fig)
print("Interpretation: train/evaluate as newly inundated flood change. Do not swap A and B.")

  target    positive_semantics permanent_water                              temporal_order  train_positive_fraction  median_A_flood  median_A_land  median_B_flood  median_B_land  density_crossover_B_gray
GT > 127 newly inundated flood negative / GT=0 A=PRE, B=POST; asymmetric; no date swapping                 0.339074           136.0          144.0            39.0          173.0                      98.5
Interpretation: train/evaluate as newly inundated flood change. Do not swap A and B.


In [27]:
# Cell 12 — S1GFloods preprocessing: SINGLE source of truth for PNG training and GeoTIFF inference

class S1GFloodsPreprocessor:
    def __init__(self, lo: float, hi: float, quantize: bool, in_channels: int, input_mode: str,
                 png_scale: float = 255.0):
        if not hi > lo:
            raise ValueError(f"Expected hi > lo, got lo={lo}, hi={hi}")
        if input_mode not in {"1ch", "3ch_replicated"}:
            raise ValueError(f"Unsupported input_mode={input_mode}")
        expected_channels = 1 if input_mode == "1ch" else 3
        if int(in_channels) != expected_channels:
            raise ValueError(f"in_channels={in_channels} inconsistent with input_mode={input_mode}")
        self.lo = float(lo); self.hi = float(hi); self.quantize = bool(quantize)
        self.in_channels = int(in_channels); self.input_mode = input_mode; self.png_scale = float(png_scale)

    def from_png_gray(self, g_uint8: np.ndarray) -> np.ndarray:
        g = np.asarray(g_uint8, dtype=np.float32)
        return np.clip(g / self.png_scale, 0.0, 1.0).astype(np.float32)

    def from_db(self, vv_db: np.ndarray, fill_gray: float = 0.0) -> Tuple[np.ndarray, np.ndarray]:
        vv = np.asarray(vv_db, dtype=np.float32)
        valid = np.isfinite(vv)
        safe = np.where(valid, vv, self.lo).astype(np.float32)
        x = np.clip((safe - self.lo) / (self.hi - self.lo), 0.0, 1.0)
        if self.quantize:
            x = np.round(x * self.png_scale) / self.png_scale
        fill_norm = float(np.clip(fill_gray / self.png_scale, 0.0, 1.0))
        x[~valid] = fill_norm
        return x.astype(np.float32), valid

    def expand_channels(self, x01: np.ndarray) -> np.ndarray:
        x = np.asarray(x01, dtype=np.float32)
        if x.ndim != 2:
            raise ValueError(f"Expected 2D gray array, got {x.shape}")
        if self.input_mode == "1ch":
            return x[None, ...]
        return np.repeat(x[None, ...], 3, axis=0)

    def png_to_model(self, g_uint8: np.ndarray) -> np.ndarray:
        return self.expand_channels(self.from_png_gray(g_uint8))

    def db_to_model(self, vv_db: np.ndarray, fill_gray: float = 0.0) -> Tuple[np.ndarray, np.ndarray]:
        x, valid = self.from_db(vv_db, fill_gray=fill_gray)
        return self.expand_channels(x), valid

    def to_dict(self) -> Dict[str, Any]:
        return {
            "lo": self.lo, "hi": self.hi, "quantize": self.quantize,
            "in_channels": self.in_channels, "input_mode": self.input_mode,
            "png_scale": self.png_scale, "expected_units": "dB", "band_name": "VV",
            "expected_resolution_m": 10,
        }

    @classmethod
    def from_dict(cls, d: Dict[str, Any]) -> "S1GFloodsPreprocessor":
        return cls(d["lo"], d["hi"], d["quantize"], d["in_channels"], d["input_mode"], d.get("png_scale", 255.0))

PREPROCESSOR = S1GFloodsPreprocessor(PNG_DB_LO, PNG_DB_HI, PNG_QUANTIZE, IN_CHANNELS, INPUT_MODE)
print("Preprocessor:", PREPROCESSOR.to_dict())
print("UNVERIFIED: checkpoint will preserve the inferred dB→gray range", (PNG_DB_LO, PNG_DB_HI))
with open(os.path.join(DIRS["configs"], "preprocessing.json"), "w") as f:
    json.dump(PREPROCESSOR.to_dict(), f, indent=2)

Preprocessor: {'lo': -30.0, 'hi': 0.0, 'quantize': True, 'in_channels': 1, 'input_mode': '1ch', 'png_scale': 255.0, 'expected_units': 'dB', 'band_name': 'VV', 'expected_resolution_m': 10}
UNVERIFIED: checkpoint will preserve the inferred dB→gray range (-30.0, 0.0)


In [28]:
# Cell 13 — Synchronized spatial + gray-domain radiometric augmentation

class SyncedSpatialAugment:
    """Apply one sampled flip/rotation identically to PRE, POST, label, and valid mask."""
    def __init__(self, enabled: bool = True):
        self.enabled = enabled

    def __call__(self, pre: np.ndarray, post: np.ndarray, label: np.ndarray, valid_mask: np.ndarray):
        if not self.enabled:
            return pre, post, label, valid_mask
        hflip = random.random() < 0.5
        vflip = random.random() < 0.5
        k = random.choice([0, 1, 2, 3])

        arrays = [pre, post, label, valid_mask]
        out = []
        for arr in arrays:
            x = arr
            if hflip: x = x[:, ::-1]
            if vflip: x = x[::-1, :]
            if k: x = np.rot90(x, k, axes=(0, 1))
            out.append(x.copy())
        return tuple(out)


class SARRadiometricAugment:
    """Training-only augmentation in S1GFloods 8-bit GRAY units. PRE/POST stay paired and are never swapped."""
    def __init__(self, enabled=True, dark_tail=True, gain_offset=True, date_offset=True, blur=True, quantize=True):
        self.enabled = enabled
        self.dark_tail = dark_tail; self.gain_offset = gain_offset
        self.date_offset = date_offset; self.blur = blur; self.quantize = quantize

    def __call__(self, pre: np.ndarray, post: np.ndarray):
        if not self.enabled:
            return pre, post
        pre = pre.astype(np.float32, copy=True); post = post.astype(np.float32, copy=True)

        # (a) Joint dark-tail compression, p=0.5: simulate external scenes with a weak/missing dark tail.
        if self.dark_tail and random.random() < 0.5:
            if random.random() < 0.5:
                f = random.uniform(0.0, 60.0)
                pre = f + pre * (255.0 - f) / 255.0
                post = f + post * (255.0 - f) / 255.0
            else:
                f = random.uniform(0.0, 85.0)
                pre = np.maximum(pre, f); post = np.maximum(post, f)

        # (b) Joint gain and offset.
        if self.gain_offset:
            gain = random.uniform(0.85, 1.15)
            offset = random.uniform(-15.0, 15.0)
            pre = pre * gain + offset; post = post * gain + offset

        # (c) Independent small per-date offsets, p=0.5.
        if self.date_offset and random.random() < 0.5:
            pre = pre + np.random.normal(0.0, 8.0)
            post = post + np.random.normal(0.0, 8.0)

        # (d) Joint blur strength (same sigma, separately applied to each date), p=0.3.
        if self.blur and random.random() < 0.3:
            sigma = random.uniform(0.5, 1.0)
            pre = gaussian_filter(pre, sigma=sigma, mode="reflect")
            post = gaussian_filter(post, sigma=sigma, mode="reflect")

        # (e) Return to legal S1GFloods gray domain.
        pre = np.clip(pre, 0.0, 255.0); post = np.clip(post, 0.0, 255.0)
        if self.quantize:
            pre = np.round(pre); post = np.round(post)
        return pre.astype(np.float32), post.astype(np.float32)

SPATIAL_AUG = SyncedSpatialAugment(enabled=USE_AUGMENTATION)
RADIOMETRIC_AUG = SARRadiometricAugment(
    enabled=USE_SAR_RADIOMETRIC_AUG,
    dark_tail=AUG_DARK_TAIL, gain_offset=AUG_GAIN_OFFSET,
    date_offset=AUG_DATE_OFFSET, blur=AUG_BLUR, quantize=PNG_QUANTIZE,
)
print("Spatial augmentation:", USE_AUGMENTATION)
print("Radiometric augmentation:", USE_SAR_RADIOMETRIC_AUG,
      {"dark_tail": AUG_DARK_TAIL, "gain_offset": AUG_GAIN_OFFSET, "date_offset": AUG_DATE_OFFSET, "blur": AUG_BLUR})

Spatial augmentation: True
Radiometric augmentation: True {'dark_tail': True, 'gain_offset': True, 'date_offset': True, 'blur': True}


In [29]:
# Cell 14 — PyTorch Dataset for S1GFloods PNG A/B/GT

class S1GFloodsChangeDataset(Dataset):
    def __init__(self, triplets: List[Dict[str, str]], preprocessor: S1GFloodsPreprocessor,
                 spatial_aug: Optional[SyncedSpatialAugment] = None,
                 radiometric_aug: Optional[SARRadiometricAugment] = None):
        self.triplets = triplets
        self.preprocessor = preprocessor
        self.spatial_aug = spatial_aug
        self.radiometric_aug = radiometric_aug

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        t = self.triplets[idx]
        pre_gray = read_png_gray(t["pre"]).astype(np.float32)
        post_gray = read_png_gray(t["post"]).astype(np.float32)
        gt = read_png_gray(t["label"])
        assert pre_gray.shape == post_gray.shape == gt.shape, (
            f"Shape mismatch for {t['key']}: pre={pre_gray.shape}, post={post_gray.shape}, GT={gt.shape}"
        )

        label = (gt > 127).astype(np.float32)
        valid_mask = np.ones_like(label, dtype=np.bool_)
        if MASK_BOTH_ZERO:
            valid_mask &= ~((pre_gray == 0) & (post_gray == 0))

        # Keep GRAY [0,255] through radiometric augmentation; normalize only at the end.
        if self.radiometric_aug is not None:
            pre_gray, post_gray = self.radiometric_aug(pre_gray, post_gray)
        if self.spatial_aug is not None:
            pre_gray, post_gray, label, valid_mask = self.spatial_aug(pre_gray, post_gray, label, valid_mask)

        pre = self.preprocessor.expand_channels(self.preprocessor.from_png_gray(pre_gray))
        post = self.preprocessor.expand_channels(self.preprocessor.from_png_gray(post_gray))
        pre = np.nan_to_num(pre, nan=0.0, posinf=1.0, neginf=0.0).astype(np.float32)
        post = np.nan_to_num(post, nan=0.0, posinf=1.0, neginf=0.0).astype(np.float32)

        return {
            "pre": torch.from_numpy(pre).float(),
            "post": torch.from_numpy(post).float(),
            "target": torch.from_numpy(label.astype(np.float32)),
            "valid_mask": torch.from_numpy(valid_mask.astype(np.bool_)),
            "key": t["key"],
        }

In [30]:
# Cell 15 — DataLoaders

TRAIN_DS = S1GFloodsChangeDataset(SPLITS["train"], PREPROCESSOR, SPATIAL_AUG, RADIOMETRIC_AUG)
VAL_DS   = S1GFloodsChangeDataset(SPLITS["val"], PREPROCESSOR, spatial_aug=None, radiometric_aug=None)
TEST_DS  = S1GFloodsChangeDataset(SPLITS["test"], PREPROCESSOR, spatial_aug=None, radiometric_aug=None)

def make_loader(ds, batch_size, shuffle, sampler=None):
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=(shuffle and sampler is None),
        sampler=sampler,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
        persistent_workers=False
    )

TRAIN_LOADER = make_loader(TRAIN_DS, BATCH_SIZE_PER_GPU, shuffle=True)
VAL_LOADER = make_loader(VAL_DS, BATCH_SIZE_PER_GPU, shuffle=False)
TEST_LOADER = make_loader(TEST_DS, BATCH_SIZE_PER_GPU, shuffle=False)

print(f"Train batches: {len(TRAIN_LOADER)} | Val batches: {len(VAL_LOADER)} | Test batches: {len(TEST_LOADER)}")
batch = next(iter(TRAIN_LOADER))
print("Sample batch shapes -> pre:", batch["pre"].shape, "post:", batch["post"].shape,
      "target:", batch["target"].shape, "valid_mask:", batch["valid_mask"].shape)
print(f"Configured micro-batch/GPU={BATCH_SIZE_PER_GPU}, ACCUM_STEPS={ACCUM_STEPS}. "
      "Effective global batch also depends on the number of active training processes/GPUs.")

Train batches: 1075 | Val batches: 133 | Test batches: 133
Sample batch shapes -> pre: torch.Size([4, 1, 256, 256]) post: torch.Size([4, 1, 256, 256]) target: torch.Size([4, 256, 256]) valid_mask: torch.Size([4, 256, 256])
Configured micro-batch/GPU=4, ACCUM_STEPS=4. Effective global batch also depends on the number of active training processes/GPUs.


In [31]:
# Cell 16 — Model zoo + factory. All model-specific logic lives here only.

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)


class UNetEncoder(nn.Module):
    def __init__(self, in_ch, base=32):
        super().__init__()
        chs = [base, base*2, base*4, base*8, base*16]
        self.enc1 = ConvBlock(in_ch, chs[0]); self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ConvBlock(chs[0], chs[1]); self.pool2 = nn.MaxPool2d(2)
        self.enc3 = ConvBlock(chs[1], chs[2]); self.pool3 = nn.MaxPool2d(2)
        self.enc4 = ConvBlock(chs[2], chs[3]); self.pool4 = nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(chs[3], chs[4])
        self.chs = chs
    def forward(self, x):
        e1 = self.enc1(x); p1 = self.pool1(e1)
        e2 = self.enc2(p1); p2 = self.pool2(e2)
        e3 = self.enc3(p2); p3 = self.pool3(e3)
        e4 = self.enc4(p3); p4 = self.pool4(e4)
        b = self.bottleneck(p4)
        return b, [e1, e2, e3, e4]


class UNetDecoder(nn.Module):
    def __init__(self, chs, out_ch=1):
        super().__init__()
        self.up4 = nn.ConvTranspose2d(chs[4], chs[3], 2, stride=2); self.dec4 = ConvBlock(chs[3]*2, chs[3])
        self.up3 = nn.ConvTranspose2d(chs[3], chs[2], 2, stride=2); self.dec3 = ConvBlock(chs[2]*2, chs[2])
        self.up2 = nn.ConvTranspose2d(chs[2], chs[1], 2, stride=2); self.dec2 = ConvBlock(chs[1]*2, chs[1])
        self.up1 = nn.ConvTranspose2d(chs[1], chs[0], 2, stride=2); self.dec1 = ConvBlock(chs[0]*2, chs[0])
        self.out_conv = nn.Conv2d(chs[0], out_ch, 1)
    def forward(self, b, skips):
        e1, e2, e3, e4 = skips
        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out_conv(d1)  # raw logits


class BiTemporalUNet(nn.Module):
    """Early-fusion baseline: concatenates PRE+POST (2*IN_CHANNELS) into a single U-Net."""
    def __init__(self, in_ch=IN_CHANNELS, base=32):
        super().__init__()
        self.encoder = UNetEncoder(in_ch=2 * in_ch, base=base)
        self.decoder = UNetDecoder(self.encoder.chs, out_ch=1)
    def forward(self, pre, post):
        x = torch.cat([pre, post], dim=1)
        b, skips = self.encoder(x)
        return self.decoder(b, skips)  # logits [B,1,H,W]


class SiameseUNetEncoder(nn.Module):
    """Weight-shared encoder applied to PRE and POST separately (Siamese)."""
    def __init__(self, in_ch=IN_CHANNELS, base=32):
        super().__init__()
        self.shared = UNetEncoder(in_ch=in_ch, base=base)
    def forward(self, pre, post):
        b_pre, s_pre = self.shared(pre)
        b_post, s_post = self.shared(post)
        return (b_pre, b_post), (s_pre, s_post)


class SiameseUNet(nn.Module):
    """Siamese U-Net with feature concatenation at bottleneck+skips (FC-Siam-conc style)."""
    def __init__(self, in_ch=IN_CHANNELS, base=32):
        super().__init__()
        self.encoder = SiameseUNetEncoder(in_ch=in_ch, base=base)
        chs = self.encoder.shared.chs
        cat_chs = [c*2 for c in chs]
        self.decoder = UNetDecoder(cat_chs, out_ch=1)
    def forward(self, pre, post):
        (b_pre, b_post), (s_pre, s_post) = self.encoder(pre, post)
        b = torch.cat([b_pre, b_post], dim=1)
        skips = [torch.cat([a, c], dim=1) for a, c in zip(s_pre, s_post)]
        return self.decoder(b, skips)


class FCSiamDiff(nn.Module):
    """FC-Siam-diff: Siamese encoder, decoder consumes |feat_pre - feat_post| (Daudt et al. style)."""
    def __init__(self, in_ch=IN_CHANNELS, base=32):
        super().__init__()
        self.encoder = SiameseUNetEncoder(in_ch=in_ch, base=base)
        chs = self.encoder.shared.chs
        self.decoder = UNetDecoder(chs, out_ch=1)
    def forward(self, pre, post):
        (b_pre, b_post), (s_pre, s_post) = self.encoder(pre, post)
        b = torch.abs(b_pre - b_post)
        skips = [torch.abs(a - c) for a, c in zip(s_pre, s_post)]
        return self.decoder(b, skips)


class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(nn.Conv2d(F_g, F_int, 1), nn.BatchNorm2d(F_int))
        self.W_x = nn.Sequential(nn.Conv2d(F_l, F_int, 1), nn.BatchNorm2d(F_int))
        self.psi = nn.Sequential(nn.Conv2d(F_int, 1, 1), nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)
    def forward(self, g, x):
        psi = self.psi(self.relu(self.W_g(g) + self.W_x(x)))
        return x * psi


class AttentionUNet(nn.Module):
    """Early-fusion input with attention gates on the skip connections."""
    def __init__(self, in_ch=IN_CHANNELS, base=32):
        super().__init__()
        self.encoder = UNetEncoder(in_ch=2 * in_ch, base=base)
        chs = self.encoder.chs
        self.att4 = AttentionGate(chs[3], chs[3], chs[3]//2)
        self.att3 = AttentionGate(chs[2], chs[2], chs[2]//2)
        self.att2 = AttentionGate(chs[1], chs[1], chs[1]//2)
        self.att1 = AttentionGate(chs[0], chs[0], chs[0]//2)
        self.up4 = nn.ConvTranspose2d(chs[4], chs[3], 2, stride=2); self.dec4 = ConvBlock(chs[3]*2, chs[3])
        self.up3 = nn.ConvTranspose2d(chs[3], chs[2], 2, stride=2); self.dec3 = ConvBlock(chs[2]*2, chs[2])
        self.up2 = nn.ConvTranspose2d(chs[2], chs[1], 2, stride=2); self.dec2 = ConvBlock(chs[1]*2, chs[1])
        self.up1 = nn.ConvTranspose2d(chs[1], chs[0], 2, stride=2); self.dec1 = ConvBlock(chs[0]*2, chs[0])
        self.out_conv = nn.Conv2d(chs[0], 1, 1)
    def forward(self, pre, post):
        x = torch.cat([pre, post], dim=1)
        b, (e1, e2, e3, e4) = self.encoder(x)
        g4 = self.up4(b); e4a = self.att4(g4, e4); d4 = self.dec4(torch.cat([g4, e4a], dim=1))
        g3 = self.up3(d4); e3a = self.att3(g3, e3); d3 = self.dec3(torch.cat([g3, e3a], dim=1))
        g2 = self.up2(d3); e2a = self.att2(g2, e2); d2 = self.dec2(torch.cat([g2, e2a], dim=1))
        g1 = self.up1(d2); e1a = self.att1(g1, e1); d1 = self.dec1(torch.cat([g1, e1a], dim=1))
        return self.out_conv(d1)


# =====================================================================================
# BIT — Bitemporal Image Transformer (Chen, Qi & Shi, 2021)
# Shared lightweight CNN backbone -> a handful of learned "semantic tokens" per image ->
# a joint transformer encoder over the concatenated pre+post tokens (context exchange) ->
# a per-image transformer decoder where every pixel attends back to the refined tokens ->
# absolute feature difference -> conv head -> upsample to full resolution.
# Adapted for 2-channel (VV, VH) SAR input instead of 3-channel optical.
# =====================================================================================

class BasicResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = None
        if stride != 1 or in_ch != out_ch:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False), nn.BatchNorm2d(out_ch))
    def forward(self, x):
        identity = x if self.downsample is None else self.downsample(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + identity)


class BITBackbone(nn.Module):
    """Lightweight ResNet-style stem, downsamples to 1/8 resolution (no ImageNet weights — SAR input)."""
    def __init__(self, in_ch=IN_CHANNELS, out_ch=32):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, 32, 7, stride=2, padding=3), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1),
        )  # 1/4
        self.layer1 = self._make_layer(32, 32, 2, stride=1)      # 1/4
        self.layer2 = self._make_layer(32, 64, 2, stride=2)      # 1/8
        self.layer3 = self._make_layer(64, out_ch, 2, stride=1)  # 1/8, channel-reduced for the tokenizer
    def _make_layer(self, in_ch, out_ch, blocks, stride):
        layers = [BasicResBlock(in_ch, out_ch, stride)]
        for _ in range(blocks - 1):
            layers.append(BasicResBlock(out_ch, out_ch, 1))
        return nn.Sequential(*layers)
    def forward(self, x):
        x = self.stem(x); x = self.layer1(x); x = self.layer2(x); x = self.layer3(x)
        return x  # [B, out_ch, H/8, W/8]


class SemanticTokenizer(nn.Module):
    """Pools the spatial feature map into L learned 'semantic tokens' via spatial attention."""
    def __init__(self, in_ch, num_tokens):
        super().__init__()
        self.attn = nn.Conv2d(in_ch, num_tokens, 1)
    def forward(self, x):
        a = self.attn(x)                                   # [B, L, H, W]
        a = rearrange(a, 'b l h w -> b l (h w)')
        a = torch.softmax(a, dim=-1)
        feat = rearrange(x, 'b c h w -> b (h w) c')
        return torch.bmm(a, feat)                           # [B, L, C]


class CrossAttentionBlock(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.norm_q = nn.LayerNorm(dim); self.norm_kv = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, dim*2), nn.GELU(), nn.Linear(dim*2, dim))
    def forward(self, q, kv):
        attn_out, _ = self.attn(self.norm_q(q), self.norm_kv(kv), self.norm_kv(kv))
        q = q + attn_out
        return q + self.mlp(self.norm2(q))


class TokenTransformerDecoder(nn.Module):
    """Every pixel (as a query) attends back to the refined context tokens (as keys/values)."""
    def __init__(self, dim, heads=8, depth=8):
        super().__init__()
        self.layers = nn.ModuleList([CrossAttentionBlock(dim, heads) for _ in range(depth)])
    def forward(self, x, tokens):
        B, C, H, W = x.shape
        q = rearrange(x, 'b c h w -> b (h w) c')
        for layer in self.layers:
            q = layer(q, tokens)
        return rearrange(q, 'b (h w) c -> b c h w', h=H, w=W)


class BIT(nn.Module):
    def __init__(self, in_ch=IN_CHANNELS, backbone_dim=32, num_tokens=4, enc_depth=1, dec_depth=8, heads=8, base=32):
        super().__init__()
        self.backbone = BITBackbone(in_ch=in_ch, out_ch=backbone_dim)
        self.tokenizer = SemanticTokenizer(backbone_dim, num_tokens)
        enc_layer = nn.TransformerEncoderLayer(d_model=backbone_dim, nhead=heads,
                                                dim_feedforward=backbone_dim*2, batch_first=True, activation="gelu")
        self.token_encoder = nn.TransformerEncoder(enc_layer, num_layers=enc_depth)
        self.decoder = TokenTransformerDecoder(backbone_dim, heads=heads, depth=dec_depth)
        self.head = nn.Sequential(
            nn.Conv2d(backbone_dim, base, 3, padding=1), nn.BatchNorm2d(base), nn.ReLU(inplace=True),
            nn.Conv2d(base, base, 3, padding=1), nn.BatchNorm2d(base), nn.ReLU(inplace=True),
            nn.Conv2d(base, 1, 1),
        )
    def forward(self, pre, post):
        H, W = pre.shape[-2:]
        f_pre, f_post = self.backbone(pre), self.backbone(post)
        t_pre, t_post = self.tokenizer(f_pre), self.tokenizer(f_post)
        tokens = self.token_encoder(torch.cat([t_pre, t_post], dim=1))
        L = t_pre.shape[1]
        t_pre, t_post = tokens[:, :L], tokens[:, L:]
        f_pre, f_post = self.decoder(f_pre, t_pre), self.decoder(f_post, t_post)
        out = self.head(torch.abs(f_pre - f_post))
        return F.interpolate(out, size=(H, W), mode="bilinear", align_corners=False)


# =====================================================================================
# SAM-CD (Ding, Zhu, Peng, Tang, Yang & Bruzzone, IEEE TGRS 62, 2024, DOI 10.1109/TGRS.2024.3368168)
# Official code: https://github.com/ggsDing/SAM-CD  (models/SAM_CD.py)
#
# Architecture-faithful port:
#   * FastSAM-x encoder (YOLOv8x-seg backbone+neck, SA-1B-pretrained), FROZEN, always in eval mode.
#     Multi-scale features: layer 15 (P3, stride 8, 320 ch), layer 18 (P4, stride 16, 640 ch),
#     layer 21 (P5, stride 32, 640 ch) and layer 1 (stride 4, 160 ch). The official repo exposes exactly these
#     tensors by patching Ultralytics source files; here they are captured by running the layers directly.
#   * Adaptors (1x1 conv+BN+ReLU) -> shared 3-stage decoder per date -> semantic-latent head (num_embed ch)
#   * Change branch: concat(decA, decB) -> 6 ResBlocks(128) -> 1x1 head(16) * SpatialAttention(latents) -> 1 logit
#   * forward(pre, post) returns ONLY the change logit [B,1,H,W] (the notebook-wide model interface);
#     the two semantic-latent maps are kept in `last_aux` for the optional TSL loss.
# =====================================================================================

def _conv1x1(in_planes, out_planes, stride=1):
    return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)

def _conv3x3(in_planes, out_planes, stride=1, groups=1, dilation=1):
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride, padding=dilation,
                     groups=groups, bias=False, dilation=dilation)


class _SpaceAttention(nn.Module):
    def __init__(self, in_channels, out_channels, reduction=4):
        super().__init__()
        self.SA = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // reduction, 1),
            nn.BatchNorm2d(in_channels // reduction, momentum=0.95),
            nn.ReLU(inplace=False),
            nn.Conv2d(in_channels // reduction, out_channels, 1),
            nn.Sigmoid(),
        )
    def forward(self, x):
        return self.SA(x)


class _SAMCDDecoderBlock(nn.Module):
    def __init__(self, in_channels_high, in_channels_low, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels_high, in_channels_high, kernel_size=2, stride=2)
        in_channels = in_channels_high + in_channels_low
        self.decode = nn.Sequential(
            _conv3x3(in_channels, out_channels), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            _conv3x3(out_channels, out_channels), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
        )
    def forward(self, x, low_feat):
        x = self.up(x)
        x = torch.cat((x, low_feat), dim=1)
        return self.decode(x)


class _SAMCDResBlock(nn.Module):
    expansion = 1
    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super().__init__()
        self.conv1 = _conv3x3(inplanes, planes, stride)
        self.bn1 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = _conv3x3(planes, planes)
        self.bn2 = nn.BatchNorm2d(planes)
        self.downsample = downsample
        self.stride = stride
    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        return self.relu(out + identity)


class FastSAMMultiScaleEncoder(nn.Module):
    """Frozen FastSAM-x encoder returning (feat_s8, feat_s16, feat_s32, feat_s4) — the official SAM-CD order."""
    TAP_LAYERS = (15, 18, 21, 1)                 # official: m.f = [15, 18, 21, 1]
    EXPECTED_CHANNELS = {15: 320, 18: 640, 21: 640, 1: 160}   # FastSAM-x (YOLOv8x-seg, width 1.25)
    HEAD_LAYER_INDEX = 22                        # the Segment head, which is NOT executed

    def __init__(self, weights_path: str):
        super().__init__()
        import ultralytics  # noqa: F401  (makes the pickled Ultralytics classes importable)
        # weights_only=False is required: the file is a pickled Ultralytics model object.
        ckpt = torch.load(weights_path, map_location="cpu", weights_only=False)
        net = None
        if isinstance(ckpt, dict):
            for k in ("ema", "model"):
                if ckpt.get(k) is not None and not isinstance(ckpt[k], dict):
                    net = ckpt[k]; break
        elif isinstance(ckpt, nn.Module):
            net = ckpt
        if net is None:
            raise RuntimeError(f"Could not find an nn.Module under 'ema'/'model' in {weights_path}.")
        net = net.float().eval()
        mods = list(net.model)
        if len(mods) != self.HEAD_LAYER_INDEX + 1 or type(mods[self.HEAD_LAYER_INDEX]).__name__ != "Segment":
            raise RuntimeError(
                f"Unexpected FastSAM architecture: {len(mods)} layers, head={type(mods[-1]).__name__}. "
                "SAM-CD expects FastSAM-x (YOLOv8x-seg: 22 backbone/neck layers + Segment head).")
        self.layers = nn.ModuleList(mods[: self.HEAD_LAYER_INDEX])     # layers 0..21; head dropped
        for i, m in enumerate(self.layers):
            if getattr(m, "i", i) != i:
                raise RuntimeError(f"Layer index mismatch at {i}: module reports i={m.i}.")
        self.keep = set(int(j) for j in getattr(net, "save", [])) | set(self.TAP_LAYERS)
        for p in self.parameters():
            p.requires_grad = False
        self.eval()

    def train(self, mode: bool = True):
        # The encoder is frozen: it must stay in eval mode (BatchNorm running stats must never change).
        return super().train(False)

    @torch.no_grad()
    def forward(self, x):
        y, taps = [], {}
        for m in self.layers:
            if m.f != -1:
                x = y[m.f] if isinstance(m.f, int) else [x if j == -1 else y[j] for j in m.f]
            x = m(x)
            y.append(x if m.i in self.keep else None)
            if m.i in self.TAP_LAYERS:
                taps[m.i] = x
        for k, c in self.EXPECTED_CHANNELS.items():
            if taps[k].shape[1] != c:
                raise RuntimeError(f"FastSAM tap layer {k} has {taps[k].shape[1]} channels, expected {c}. "
                                   "Only FastSAM-x is supported.")
        return taps[15], taps[18], taps[21], taps[1]      # s8, s16, s32, s4


def _samcd_init_weights(*models):
    """Identical to the official utils.misc.initialize_weights (ConvTranspose2d keeps PyTorch defaults)."""
    for model in models:
        for module in model.modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)):
                nn.init.kaiming_normal_(module.weight)
                if module.bias is not None:
                    module.bias.data.zero_()
            elif isinstance(module, nn.BatchNorm2d):
                module.weight.data.fill_(1)
                module.bias.data.zero_()


class SAMCD(nn.Module):
    def __init__(self, weights_path: str, num_embed: int = 8):
        super().__init__()
        self.encoder = FastSAMMultiScaleEncoder(weights_path)

        self.Adapter32 = nn.Sequential(nn.Conv2d(640, 160, 1, 1, 0, bias=False), nn.BatchNorm2d(160), nn.ReLU())
        self.Adapter16 = nn.Sequential(nn.Conv2d(640, 160, 1, 1, 0, bias=False), nn.BatchNorm2d(160), nn.ReLU())
        self.Adapter8 = nn.Sequential(nn.Conv2d(320, 80, 1, 1, 0, bias=False), nn.BatchNorm2d(80), nn.ReLU())
        self.Adapter4 = nn.Sequential(nn.Conv2d(160, 40, 1, 1, 0, bias=False), nn.BatchNorm2d(40), nn.ReLU())

        self.Dec2 = _SAMCDDecoderBlock(160, 160, 80)
        self.Dec1 = _SAMCDDecoderBlock(80, 80, 40)
        self.Dec0 = _SAMCDDecoderBlock(40, 40, 64)

        self.SA = _SpaceAttention(16, 16, 4)
        self.segmenter = nn.Conv2d(64, num_embed, kernel_size=1)
        self.resCD = self._make_layer(_SAMCDResBlock, 128, 128, 6, stride=1)
        self.headC = nn.Sequential(nn.Conv2d(128, 16, 1, 1, 0, bias=False), nn.BatchNorm2d(16), nn.ReLU())
        self.segmenterC = nn.Conv2d(16, 1, kernel_size=1)

        _samcd_init_weights(self.Adapter32, self.Adapter16, self.Adapter8, self.Adapter4, self.Dec2, self.Dec1,
                            self.Dec0, self.segmenter, self.resCD, self.headC, self.segmenterC)

    @staticmethod
    def _make_layer(block, inplanes, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or inplanes != planes:
            downsample = nn.Sequential(_conv1x1(inplanes, planes, stride), nn.BatchNorm2d(planes))
        layers = [block(inplanes, planes, stride, downsample)]
        for _ in range(1, blocks):
            layers.append(block(planes * block.expansion, planes))
        return nn.Sequential(*layers)

    def _decode_one(self, f8, f16, f32, f4):
        a4 = self.Adapter4(f4.clone()); a8 = self.Adapter8(f8.clone())
        a16 = self.Adapter16(f16.clone()); a32 = self.Adapter32(f32.clone())
        d2 = self.Dec2(a32, a16); d1 = self.Dec1(d2, a8); d0 = self.Dec0(d1, a4)
        return d0, self.segmenter(d0)

    def forward(self, x1, x2):
        input_shape = x1.shape[-2:]
        # Frozen encoder in eval mode => running both dates in one batch is exactly equivalent to two calls.
        n = x1.shape[0]
        f8, f16, f32, f4 = self.encoder(torch.cat([x1, x2], dim=0))
        fA = (f8[:n], f16[:n], f32[:n], f4[:n])
        fB = (f8[n:], f16[n:], f32[n:], f4[n:])
        # Trainable layers are applied per date (separate BatchNorm batch statistics), as in the official code.
        decA_0, outA = self._decode_one(*fA)
        decB_0, outB = self._decode_one(*fB)

        A = self.SA(torch.cat([outA, outB], dim=1))
        featC = self.resCD(torch.cat([decA_0, decB_0], dim=1))
        featC = self.headC(featC) * A
        outC = self.segmenterC(featC)
        up = lambda t: F.interpolate(t, input_shape, mode="bilinear", align_corners=True)
        return up(outC), up(outA), up(outB)


class SemanticLatentSimilarityLoss(nn.Module):
    """Official SAM-CD `LatentSimilarity` (utils/loss.py): softmax(latent/T) of both dates, cosine embedding loss where
    UNCHANGED pixels are pulled together and changed pixels contribute 0 (the official target is {0,1}, which
    nn.CosineEmbeddingLoss treats as 'no term' for 0). Valid-pixel masked; identical to the official value when all
    pixels are valid."""
    def __init__(self, T: float = 3.0, weight: float = 1.0):
        super().__init__()
        self.T, self.weight = float(T), float(weight)
    def forward(self, lat_a, lat_b, target, valid_mask):
        pa = F.softmax(lat_a.float() / self.T, dim=1)
        pb = F.softmax(lat_b.float() / self.T, dim=1)
        cos = F.cosine_similarity(pa, pb, dim=1, eps=1e-8)                 # [B,H,W]
        unchanged = (target.squeeze(1) < 0.5).float()
        valid = valid_mask.squeeze(1).float()
        loss = ((1.0 - cos) * unchanged * valid).sum() / valid.sum().clamp_min(1.0)
        return self.weight * loss


class SAMCDWrapper(nn.Module):
    """Notebook-wide interface: forward(pre, post) -> change logits [B,1,H,W]."""
    def __init__(self, weights_path, in_chans=IN_CHANNELS, num_embed=8, use_tsl=False, tsl_T=3.0, tsl_weight=1.0):
        super().__init__()
        if in_chans not in (1, 3):
            raise ValueError("SAM-CD wrapper supports IN_CHANNELS in {1, 3}.")
        self.in_chans = int(in_chans)
        self.net = SAMCD(weights_path, num_embed=num_embed)
        self.use_tsl = bool(use_tsl)
        self.aux_loss_fn = SemanticLatentSimilarityLoss(tsl_T, tsl_weight) if self.use_tsl else None
        self.last_aux = None
    def _to_rgb(self, x):
        # FastSAM expects 3 channels in [0,1]; the single VV gray channel is replicated (same as the repo's image/255).
        return x.repeat(1, 3, 1, 1) if x.shape[1] == 1 else x
    def forward(self, pre, post):
        out_c, out_a, out_b = self.net(self._to_rgb(pre), self._to_rgb(post))
        self.last_aux = (out_a, out_b) if (self.training and self.use_tsl) else None
        return out_c


MODEL_REGISTRY = {
    "bitemporal_unet": lambda cfg: BiTemporalUNet(in_ch=cfg.get("IN_CHANNELS", IN_CHANNELS), base=cfg.get("unet_base", 32)),
    "siamese_unet":    lambda cfg: SiameseUNet(in_ch=cfg.get("IN_CHANNELS", IN_CHANNELS), base=cfg.get("unet_base", 32)),
    "fc_siam_conc":    lambda cfg: SiameseUNet(in_ch=cfg.get("IN_CHANNELS", IN_CHANNELS), base=cfg.get("unet_base", 32)),
    "fc_siam_diff":    lambda cfg: FCSiamDiff(in_ch=cfg.get("IN_CHANNELS", IN_CHANNELS), base=cfg.get("unet_base", 32)),
    "attention_unet":  lambda cfg: AttentionUNet(in_ch=cfg.get("IN_CHANNELS", IN_CHANNELS), base=cfg.get("unet_base", 32)),
    "bit": lambda cfg: BIT(
        in_ch=cfg.get("IN_CHANNELS", IN_CHANNELS), backbone_dim=cfg.get("BIT_DIM", 32), num_tokens=cfg.get("BIT_TOKENS", 4),
        enc_depth=cfg.get("BIT_ENC_DEPTH", 1), dec_depth=cfg.get("BIT_DEC_DEPTH", 8),
        heads=cfg.get("BIT_HEADS", 8), base=cfg.get("unet_base", 32)),
    "sam_cd": lambda cfg: SAMCDWrapper(
        weights_path=cfg.get("SAMCD_WEIGHTS", FASTSAM_PATH),
        in_chans=cfg.get("IN_CHANNELS", IN_CHANNELS),
        num_embed=cfg.get("SAMCD_NUM_EMBED", 8),
        use_tsl=cfg.get("SAMCD_USE_TSL", False),
        tsl_T=cfg.get("SAMCD_TSL_TEMPERATURE", 3.0),
        tsl_weight=cfg.get("SAMCD_TSL_WEIGHT", 1.0)),
}

def build_model(config: Dict[str, Any]) -> nn.Module:
    name = config.get("MODEL_NAME", MODEL_NAME)
    if name not in MODEL_REGISTRY:
        raise ValueError(f"Unknown MODEL_NAME '{name}'. Available: {list(MODEL_REGISTRY.keys())}")
    model = MODEL_REGISTRY[name](config)
    return model

_test_model = build_model(CONFIG)
n_params = sum(p.numel() for p in _test_model.parameters())
n_trainable = sum(p.numel() for p in _test_model.parameters() if p.requires_grad)
print(f"Built '{MODEL_NAME}': total {n_params:,} | trainable {n_trainable:,} | frozen {n_params - n_trainable:,} parameters.")
if MODEL_NAME == "sam_cd":
    # Fail early (before any training) if the FastSAM hooks / channel counts / shapes are wrong.
    _test_model.eval()
    with torch.no_grad():
        _x = torch.zeros(1, IN_CHANNELS, 256, 256)
        _o = _test_model(_x, _x)
    assert tuple(_o.shape) == (1, 1, 256, 256), f"Unexpected SAM-CD output shape {tuple(_o.shape)}"
    _enc_bn_train = any(m.training for m in _test_model.net.encoder.modules())
    _test_model.train()
    assert not any(m.training for m in _test_model.net.encoder.modules()), "FastSAM encoder must stay in eval mode"
    assert not any(p.requires_grad for p in _test_model.net.encoder.parameters()), "FastSAM encoder must be frozen"
    print("SAM-CD dummy forward OK:", tuple(_o.shape), "| encoder frozen & eval-locked: True")
del _test_model


Built 'sam_cd': total 62,025,997 | trainable 2,591,357 | frozen 59,434,640 parameters.
SAM-CD dummy forward OK: (1, 1, 256, 256) | encoder frozen & eval-locked: True


In [32]:
# Cell 17 — Losses (all masked to valid pixels)

def masked_bce(logits, target, valid_mask, eps=1e-7):
    loss = F.binary_cross_entropy_with_logits(logits, target, reduction="none")
    loss = loss * valid_mask.float()
    denom = valid_mask.float().sum().clamp_min(1.0)
    return loss.sum() / denom

def masked_dice(logits, target, valid_mask, eps=1e-6):
    prob = torch.sigmoid(logits)
    prob = prob * valid_mask.float(); tgt = target * valid_mask.float()
    inter = (prob * tgt).sum(dim=(1, 2, 3))
    union = prob.sum(dim=(1, 2, 3)) + tgt.sum(dim=(1, 2, 3))
    dice = (2 * inter + eps) / (union + eps)
    return 1 - dice.mean()

def masked_focal(logits, target, valid_mask, gamma=2.0, alpha=0.25, eps=1e-7):
    prob = torch.sigmoid(logits)
    ce = F.binary_cross_entropy_with_logits(logits, target, reduction="none")
    p_t = prob * target + (1 - prob) * (1 - target)
    alpha_t = alpha * target + (1 - alpha) * (1 - target)
    loss = alpha_t * (1 - p_t).clamp_min(eps).pow(gamma) * ce
    loss = loss * valid_mask.float()
    denom = valid_mask.float().sum().clamp_min(1.0)
    return loss.sum() / denom

def masked_tversky(logits, target, valid_mask, alpha=0.3, beta=0.7, eps=1e-6):
    prob = torch.sigmoid(logits)
    prob = prob * valid_mask.float(); tgt = target * valid_mask.float()
    tp = (prob * tgt).sum(dim=(1, 2, 3))
    fp = (prob * (1 - tgt)).sum(dim=(1, 2, 3))
    fn = ((1 - prob) * tgt).sum(dim=(1, 2, 3))
    tversky = (tp + eps) / (tp + alpha * fp + beta * fn + eps)
    return 1 - tversky.mean()

def build_loss(config: Dict[str, Any]):
    name = config.get("LOSS_NAME", LOSS_NAME)
    bce_w, dice_w = config.get("BCE_WEIGHT", 0.5), config.get("DICE_WEIGHT", 0.5)
    gamma, alpha = config.get("FOCAL_GAMMA", 2.0), config.get("FOCAL_ALPHA", 0.25)
    t_a, t_b = config.get("TVERSKY_ALPHA", 0.3), config.get("TVERSKY_BETA", 0.7)

    def loss_fn(logits, target, valid_mask):
        if name == "bce":
            return masked_bce(logits, target, valid_mask)
        if name == "dice":
            return masked_dice(logits, target, valid_mask)
        if name == "bce_dice":
            return bce_w * masked_bce(logits, target, valid_mask) + dice_w * masked_dice(logits, target, valid_mask)
        if name == "focal":
            return masked_focal(logits, target, valid_mask, gamma, alpha)
        if name == "focal_dice":
            return masked_focal(logits, target, valid_mask, gamma, alpha) + masked_dice(logits, target, valid_mask)
        if name == "tversky":
            return masked_tversky(logits, target, valid_mask, t_a, t_b)
        if name == "bce_tversky":
            return bce_w * masked_bce(logits, target, valid_mask) + dice_w * masked_tversky(logits, target, valid_mask, t_a, t_b)
        if name == "focal_tversky":
            return masked_focal(logits, target, valid_mask, gamma, alpha) + masked_tversky(logits, target, valid_mask, t_a, t_b)
        raise ValueError(f"Unknown LOSS_NAME '{name}'")
    return loss_fn

LOSS_FN = build_loss(CONFIG)

# Sanity check: invalid pixels must not affect the loss.
with torch.no_grad():
    logits = torch.randn(2, 1, 16, 16)
    target = torch.randint(0, 2, (2, 1, 16, 16)).float()
    vmask_all_valid = torch.ones(2, 1, 16, 16, dtype=torch.bool)
    vmask_some_invalid = vmask_all_valid.clone()
    vmask_some_invalid[:, :, :4, :4] = False
    target_perturbed = target.clone()
    target_perturbed[:, :, :4, :4] = 1 - target_perturbed[:, :, :4, :4]  # corrupt only the invalid region
    l1 = LOSS_FN(logits, target, vmask_some_invalid)
    l2 = LOSS_FN(logits, target_perturbed, vmask_some_invalid)
    assert torch.allclose(l1, l2, atol=1e-5), "FAIL: invalid pixels are leaking into the loss!"
    print(f"Sanity check passed: corrupting invalid-region targets changed loss by {abs((l1-l2).item()):.2e} (~0).")
print(f"Using loss: {LOSS_NAME}")


Sanity check passed: corrupting invalid-region targets changed loss by 0.00e+00 (~0).
Using loss: bce_dice


In [33]:
# Cell 18 — Metrics (valid-pixel only; accumulated confusion counts for dataset-level metrics)

@dataclass
class ConfusionAccumulator:
    tp: int = 0; fp: int = 0; fn: int = 0; tn: int = 0

    def update(self, pred_bin: np.ndarray, target: np.ndarray, valid_mask: np.ndarray):
        p = pred_bin[valid_mask].astype(bool); t = target[valid_mask].astype(bool)
        self.tp += int(np.sum(p & t)); self.fp += int(np.sum(p & ~t))
        self.fn += int(np.sum(~p & t)); self.tn += int(np.sum(~p & ~t))

    def compute(self, eps=1e-9) -> Dict[str, float]:
        tp, fp, fn, tn = self.tp, self.fp, self.fn, self.tn
        n = tp + fp + fn + tn
        iou = tp / (tp + fp + fn + eps)
        dice = 2 * tp / (2 * tp + fp + fn + eps)
        precision = tp / (tp + fp + eps)
        recall = tp / (tp + fn + eps)
        specificity = tn / (tn + fp + eps)
        accuracy = (tp + tn) / (n + eps)
        balanced_acc = 0.5 * (recall + specificity)
        return {"iou": iou, "dice_f1": dice, "precision": precision, "recall": recall,
                "specificity": specificity, "accuracy": accuracy, "balanced_accuracy": balanced_acc,
                "tp": tp, "fp": fp, "fn": fn, "tn": tn}


def per_image_metrics(prob: np.ndarray, target: np.ndarray, valid_mask: np.ndarray, threshold: float) -> Dict[str, float]:
    pred_bin = (prob >= threshold)
    acc = ConfusionAccumulator(); acc.update(pred_bin, target, valid_mask)
    m = acc.compute()
    y_true = target[valid_mask].astype(int); y_prob = prob[valid_mask]
    try:
        m["kappa"] = cohen_kappa_score(y_true, pred_bin[valid_mask].astype(int)) if len(np.unique(y_true)) > 1 else np.nan
        m["mcc"] = matthews_corrcoef(y_true, pred_bin[valid_mask].astype(int)) if len(np.unique(y_true)) > 1 else np.nan
        m["pr_auc"] = average_precision_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan
        m["roc_auc"] = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan
    except Exception:
        m["kappa"] = m["mcc"] = m["pr_auc"] = m["roc_auc"] = np.nan
    return m

print("Metrics module ready: ConfusionAccumulator, per_image_metrics.")


Metrics module ready: ConfusionAccumulator, per_image_metrics.


In [34]:
# Cell 19 — Checkpoint save/load utilities

def save_checkpoint(path, model, optimizer, scheduler, scaler, epoch, best_val_score, seed, threshold, config):
    state = model.module.state_dict() if hasattr(model, "module") else model.state_dict()
    ckpt = {
        "model_state_dict": state,
        "optimizer_state_dict": optimizer.state_dict() if optimizer is not None else None,
        "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
        "amp_scaler_state_dict": scaler.state_dict() if scaler is not None else None,
        "epoch": epoch,
        "best_val_score": best_val_score,
        "model_name": config.get("MODEL_NAME", MODEL_NAME),
        "config": config,
        "in_channels": int(config.get("IN_CHANNELS", IN_CHANNELS)),
        "seed": seed,
        "threshold": float(threshold),
        "preprocessing": PREPROCESSOR.to_dict(),
    }
    torch.save(ckpt, path)

def load_checkpoint(path, device="cpu"):
    return torch.load(path, map_location=device)

def load_model_from_checkpoint(path, device):
    ckpt = load_checkpoint(path, device)
    cfg = ckpt.get("config", CONFIG)
    model = build_model(cfg).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    return model, ckpt

# Sanity test: save/load roundtrip
_m = build_model(CONFIG)
_opt = torch.optim.AdamW(_m.parameters(), lr=1e-4)
_test_path = os.path.join(DIRS["checkpoints"], "_sanity_test.pt")
save_checkpoint(_test_path, _m, _opt, None, None, epoch=0, best_val_score=0.0, seed=SEED, threshold=0.5, config=CONFIG)
_m2, _ckpt2 = load_model_from_checkpoint(_test_path, "cpu")
assert _ckpt2["preprocessing"] == PREPROCESSOR.to_dict()
assert _ckpt2["in_channels"] == IN_CHANNELS
assert all(torch.equal(a, b) for a, b in zip(_m.state_dict().values(), _m2.state_dict().values()))
os.remove(_test_path)
print("Checkpoint save/load sanity test passed.")

Checkpoint save/load sanity test passed.


In [35]:
# Cell 20 — Train / validation epoch functions with gradient accumulation

def run_epoch(model, loader, optimizer, scaler, loss_fn, device, train: bool, use_amp: bool, accum_steps: int = ACCUM_STEPS):
    model.train(mode=train)
    total_loss, n_batches = 0.0, 0
    acc = ConfusionAccumulator()
    accum_steps = max(1, int(accum_steps)) if train else 1
    ctx = torch.enable_grad() if train else torch.no_grad()

    if train:
        optimizer.zero_grad(set_to_none=True)

    with ctx:
        for batch_idx, batch in enumerate(loader):
            pre = batch["pre"].to(device, non_blocking=True)
            post = batch["post"].to(device, non_blocking=True)
            target = batch["target"].unsqueeze(1).to(device, non_blocking=True)
            valid_mask = batch["valid_mask"].unsqueeze(1).to(device, non_blocking=True)

            with autocast(device_type="cuda", enabled=use_amp and device.type == "cuda"):
                logits = model(pre, post)
                raw_loss = loss_fn(logits, target, valid_mask)
                # Optional SAM-CD task-agnostic semantic loss (SAMCD_USE_TSL). aux_loss_fn is None for every other
                # model and for SAM-CD with the default SAMCD_USE_TSL=False, so the baseline loss is untouched.
                _core = model.module if hasattr(model, "module") else model
                if train and getattr(_core, "aux_loss_fn", None) is not None and getattr(_core, "last_aux", None) is not None:
                    raw_loss = raw_loss + _core.aux_loss_fn(_core.last_aux[0], _core.last_aux[1], target, valid_mask)
                    _core.last_aux = None
                loss_for_backward = raw_loss / accum_steps

            if train:
                if use_amp:
                    scaler.scale(loss_for_backward).backward()
                else:
                    loss_for_backward.backward()

                should_step = ((batch_idx + 1) % accum_steps == 0) or ((batch_idx + 1) == len(loader))
                if should_step:
                    if use_amp:
                        if GRAD_CLIP_NORM is not None:
                            scaler.unscale_(optimizer)
                            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                        scaler.step(optimizer); scaler.update()
                    else:
                        if GRAD_CLIP_NORM is not None:
                            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                        optimizer.step()
                    optimizer.zero_grad(set_to_none=True)

            with torch.no_grad():
                prob = torch.sigmoid(logits)
                pred_bin = (prob >= THRESHOLD).cpu().numpy().astype(bool)
                acc.update(pred_bin, target.cpu().numpy().astype(bool), valid_mask.cpu().numpy().astype(bool))

            total_loss += raw_loss.item(); n_batches += 1

    metrics = acc.compute()
    metrics["loss"] = total_loss / max(n_batches, 1)
    return metrics

print(f"run_epoch() ready with ACCUM_STEPS={ACCUM_STEPS}.")

run_epoch() ready with ACCUM_STEPS=4.


In [36]:
# Cell 21 — Multi-GPU runner (DDP with automatic DataParallel / single-GPU fallback)

def build_optimizer_scheduler(model, config, steps_per_epoch):
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["LEARNING_RATE"], weight_decay=config["WEIGHT_DECAY"])
    if config.get("SCHEDULER_NAME", "cosine") == "cosine":
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["EPOCHS"])
    else:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=5)
    return optimizer, scheduler

def ddp_worker(rank, world_size, seed, config, result_queue):
    os.environ.setdefault("MASTER_ADDR", "127.0.0.1")
    os.environ.setdefault("MASTER_PORT", "29500")
    dist.init_process_group("nccl", rank=rank, world_size=world_size)
    torch.cuda.set_device(rank)
    device = torch.device(f"cuda:{rank}")
    set_global_seed(seed)

    model = build_model(config).to(device)
    model = TorchDDP(model, device_ids=[rank])

    train_sampler = DistributedSampler(TRAIN_DS, num_replicas=world_size, rank=rank, shuffle=True, seed=seed)
    train_loader = make_loader(TRAIN_DS, config["BATCH_SIZE_PER_GPU"], shuffle=True, sampler=train_sampler)
    val_loader = VAL_LOADER  # validated on rank 0 only, below

    optimizer, scheduler = build_optimizer_scheduler(model, config, len(train_loader))
    scaler = GradScaler(
        "cuda",
        enabled=config["USE_AMP"] and device.type == "cuda"
    )
    loss_fn = build_loss(config)

    best_val_iou, patience_ctr, history = -1.0, 0, []
    for epoch in range(config["EPOCHS"]):
        train_sampler.set_epoch(epoch)
        t0 = time.time()
        train_m = run_epoch(model, train_loader, optimizer, scaler, loss_fn, device, train=True, use_amp=config["USE_AMP"], accum_steps=config.get("ACCUM_STEPS", ACCUM_STEPS))
        dist.barrier()

        if rank == 0:
            val_m = run_epoch(model, val_loader, None, None, loss_fn, device, train=False, use_amp=config["USE_AMP"], accum_steps=1)
            if config.get("SCHEDULER_NAME") == "plateau":
                scheduler.step(val_m["iou"])
            else:
                scheduler.step()
            epoch_time = time.time() - t0
            row = {"epoch": epoch, "train_loss": train_m["loss"], "val_loss": val_m["loss"],
                   "train_iou": train_m["iou"], "val_iou": val_m["iou"],
                   "train_dice": train_m["dice_f1"], "val_dice": val_m["dice_f1"],
                   "lr": optimizer.param_groups[0]["lr"], "epoch_time_sec": epoch_time}
            history.append(row)
            save_checkpoint(os.path.join(DIRS["checkpoints"], "latest_model.pt"), model, optimizer, scheduler,
                             scaler, epoch, best_val_iou, seed, THRESHOLD, config)
            if val_m["iou"] > best_val_iou:
                best_val_iou = val_m["iou"]; patience_ctr = 0
                save_checkpoint(os.path.join(DIRS["checkpoints"], "best_model.pt"), model, optimizer, scheduler,
                                 scaler, epoch, best_val_iou, seed, THRESHOLD, config)
            else:
                patience_ctr += 1
            print(f"[epoch {epoch:03d}] train_loss={train_m['loss']:.4f} val_loss={val_m['loss']:.4f} "
                  f"train_iou={train_m['iou']:.4f} val_iou={val_m['iou']:.4f} best={best_val_iou:.4f} "
                  f"patience={patience_ctr}/{config['EARLY_STOPPING_PATIENCE']} time={epoch_time:.1f}s")
            if patience_ctr >= config["EARLY_STOPPING_PATIENCE"]:
                print("Early stopping triggered."); stop_flag = torch.tensor(1, device=device)
            else:
                stop_flag = torch.tensor(0, device=device)
        else:
            stop_flag = torch.tensor(0, device=device)
        dist.broadcast(stop_flag, src=0)
        if stop_flag.item() == 1:
            break

    if rank == 0:
        pd.DataFrame(history).to_csv(os.path.join(DIRS["logs"], "training_history.csv"), index=False)
        result_queue.put(history)
    dist.destroy_process_group()


def train_single_process(config, device, seed):
    """Fallback path: single GPU, or nn.DataParallel across multiple GPUs on one process."""
    model = build_model(config).to(device)
    if EFFECTIVE_DP:
        model = nn.DataParallel(model)
        print(f"Using nn.DataParallel across {N_GPUS} GPUs.")
    effective_bs = config["BATCH_SIZE_PER_GPU"] * (N_GPUS if EFFECTIVE_DP else 1)
    train_loader = make_loader(TRAIN_DS, effective_bs, shuffle=True)

    optimizer, scheduler = build_optimizer_scheduler(model, config, len(train_loader))
    scaler = GradScaler(
        "cuda",
        enabled=config["USE_AMP"] and device.type == "cuda"
    )
    loss_fn = build_loss(config)

    start_epoch = 0
    best_val_iou = -1.0
    if config.get("RESUME_CHECKPOINT"):
        ckpt = load_checkpoint(config["RESUME_CHECKPOINT"], device)
        (model.module if hasattr(model, "module") else model).load_state_dict(ckpt["model_state_dict"])
        if ckpt.get("optimizer_state_dict"): optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        if ckpt.get("scheduler_state_dict") and scheduler is not None:
            try: scheduler.load_state_dict(ckpt["scheduler_state_dict"])
            except Exception: pass
        start_epoch = ckpt.get("epoch", 0) + 1
        best_val_iou = ckpt.get("best_val_score", -1.0)
        print(f"Resumed from {config['RESUME_CHECKPOINT']} at epoch {start_epoch}, best_val_iou={best_val_iou:.4f}")

    history, patience_ctr = [], 0
    train_wall_t0 = time.time()
    for epoch in range(start_epoch, config["EPOCHS"]):
        t0 = time.time()
        train_m = run_epoch(model, train_loader, optimizer, scaler, loss_fn, device, train=True,
                             use_amp=config["USE_AMP"], accum_steps=config.get("ACCUM_STEPS", ACCUM_STEPS))
        val_m = run_epoch(model, VAL_LOADER, None, None, loss_fn, device, train=False, use_amp=config["USE_AMP"], accum_steps=1)
        if config.get("SCHEDULER_NAME") == "plateau":
            scheduler.step(val_m["iou"])
        else:
            scheduler.step()
        epoch_time = time.time() - t0
        row = {"epoch": epoch, "train_loss": train_m["loss"], "val_loss": val_m["loss"],
               "train_iou": train_m["iou"], "val_iou": val_m["iou"],
               "train_dice": train_m["dice_f1"], "val_dice": val_m["dice_f1"],
               "lr": optimizer.param_groups[0]["lr"], "epoch_time_sec": epoch_time}
        history.append(row)
        pd.DataFrame(history).to_csv(os.path.join(DIRS["logs"], "training_history.csv"), index=False)  # crash-safe log
        save_checkpoint(os.path.join(DIRS["checkpoints"], "latest_model.pt"), model, optimizer, scheduler,
                         scaler, epoch, best_val_iou, seed, THRESHOLD, config)
        if val_m["iou"] > best_val_iou:
            best_val_iou = val_m["iou"]; patience_ctr = 0
            save_checkpoint(os.path.join(DIRS["checkpoints"], "best_model.pt"), model, optimizer, scheduler,
                             scaler, epoch, best_val_iou, seed, THRESHOLD, config)
        else:
            patience_ctr += 1
        print(f"[epoch {epoch:03d}] train_loss={train_m['loss']:.4f} val_loss={val_m['loss']:.4f} "
              f"train_iou={train_m['iou']:.4f} val_iou={val_m['iou']:.4f} best={best_val_iou:.4f} "
              f"patience={patience_ctr}/{config['EARLY_STOPPING_PATIENCE']} time={epoch_time:.1f}s")
        if patience_ctr >= config["EARLY_STOPPING_PATIENCE"]:
            print("Early stopping triggered."); break
        _max_h = config.get("MAX_TRAIN_HOURS")
        if _max_h is not None and (time.time() - train_wall_t0) / 3600.0 >= float(_max_h):
            print(f"MAX_TRAIN_HOURS={_max_h} reached after epoch {epoch}: stopping cleanly so evaluation/export can run. "
                  "NOTE: the cosine schedule did not complete; report this.")
            break

    pd.DataFrame(history).to_csv(os.path.join(DIRS["logs"], "training_history.csv"), index=False)
    return history

def build_optimizer_scheduler(model, config, steps_per_epoch):
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["LEARNING_RATE"], weight_decay=config["WEIGHT_DECAY"])
    warmup_epochs = config.get("WARMUP_EPOCHS", 0)
    if warmup_epochs > 0:
        warmup = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=warmup_epochs)
        cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["EPOCHS"] - warmup_epochs)
        scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[warmup_epochs])
    elif config.get("SCHEDULER_NAME", "cosine") == "cosine":
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["EPOCHS"])
    else:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=5)
    return optimizer, scheduler


In [37]:
# Cell 22 — Launch training for RUN_SEEDS[0] (development run). Multi-seed loop is in Cell 32.
# NOTE: the Siamese-UNet notebook forced EFFECTIVE_DDP=False and EFFECTIVE_DP=False (Cell 6), i.e. single-GPU training;
# the same strategy is used here so the benchmark is like-for-like. The DDP branch below is therefore skipped.

def train_one_seed(seed: int, config: Dict[str, Any]) -> List[Dict[str, Any]]:
    set_global_seed(seed)
    if EFFECTIVE_DDP:
        try:
            world_size = N_GPUS
            ctx = mp.get_context("spawn")
            result_queue = ctx.Queue()
            mp.spawn(ddp_worker, args=(world_size, seed, config, result_queue), nprocs=world_size, join=True)
            history = result_queue.get() if not result_queue.empty() else \
                pd.read_csv(os.path.join(DIRS["logs"], "training_history.csv")).to_dict("records")
            return history
        except Exception as e:
            print(f"DDP launch failed ({e}); falling back to single-process "
                  f"{'DataParallel' if N_GPUS > 1 else 'single-GPU'} training.")
    return train_single_process(config, DEVICE, seed)

HISTORY = train_one_seed(RUN_SEEDS[0], CONFIG)
print(f"Training complete for seed {RUN_SEEDS[0]}. Epochs run: {len(HISTORY)}.")


[epoch 000] train_loss=0.5355 val_loss=0.4521 train_iou=0.5565 val_iou=0.6823 best=0.6823 patience=0/50 time=102.6s
[epoch 001] train_loss=0.4480 val_loss=0.4292 train_iou=0.6867 val_iou=0.6606 best=0.6823 patience=1/50 time=101.6s
[epoch 002] train_loss=0.4126 val_loss=0.3777 train_iou=0.7211 val_iou=0.7243 best=0.7243 patience=0/50 time=102.0s
[epoch 003] train_loss=0.3755 val_loss=0.3560 train_iou=0.7678 val_iou=0.7676 best=0.7676 patience=0/50 time=101.4s
[epoch 004] train_loss=0.3427 val_loss=0.3121 train_iou=0.7932 val_iou=0.7859 best=0.7859 patience=0/50 time=100.9s
[epoch 005] train_loss=0.3084 val_loss=0.2498 train_iou=0.8086 val_iou=0.8235 best=0.8235 patience=0/50 time=101.2s
[epoch 006] train_loss=0.2757 val_loss=0.2201 train_iou=0.8201 val_iou=0.8497 best=0.8497 patience=0/50 time=102.2s
[epoch 007] train_loss=0.2484 val_loss=0.2190 train_iou=0.8319 val_iou=0.8269 best=0.8497 patience=1/50 time=101.8s
[epoch 008] train_loss=0.2284 val_loss=0.1966 train_iou=0.8361 val_iou=0

In [ ]:
# Cell 23 — Training curves

hist_df = pd.read_csv(os.path.join(DIRS["logs"], "training_history.csv"))

def plot_curve(df, cols, ylabel, title, fname):
    fig, ax = plt.subplots(figsize=(8, 5))
    for c in cols:
        ax.plot(df["epoch"], df[c], label=c)
    ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel); ax.set_title(title); ax.legend()
    for ext in ["png", "pdf"]:
        fig.savefig(os.path.join(DIRS["figures"], f"{fname}.{ext}"), dpi=200, bbox_inches="tight")
    plt.close(fig)

plot_curve(hist_df, ["train_loss", "val_loss"], "Loss", "Training / Validation Loss", "loss_curve")
plot_curve(hist_df, ["train_iou", "val_iou"], "IoU", "Training / Validation IoU", "iou_curve")
plot_curve(hist_df, ["train_dice", "val_dice"], "Dice / F1", "Training / Validation Dice", "dice_curve")
plot_curve(hist_df, ["lr"], "Learning Rate", "Learning Rate Schedule", "lr_curve")
print("Saved loss/IoU/Dice/LR curves (PNG+PDF) to", DIRS["figures"])
hist_df.tail()


Saved loss/IoU/Dice/LR curves (PNG+PDF) to /kaggle/working/samcd_s1gfloods/figures


,epoch,train_loss,val_loss,train_iou,val_iou,train_dice,val_dice,lr,epoch_time_sec
179,179,0.090496,0.116969,0.908042,0.896393,0.951805,0.945366,0.000003,99.102574
180,180,0.090082,0.122597,0.908618,0.893504,0.952121,0.943757,0.000002,100.093772
181,181,0.090141,0.118301,0.908623,0.897217,0.952124,0.945824,0.000002,99.518670
182,182,0.090190,0.118624,0.908150,0.895302,0.951864,0.944759,0.000002,100.281239
183,183,0.090579,0.113335,0.908030,0.900636,0.951798,0.947721,0.000002,100.716738


In [41]:
# Cell 24 — Load best checkpoint for evaluation

BEST_CKPT_PATH = os.path.join(DIRS["checkpoints"], "best_model.pt")
BEST_MODEL, BEST_CKPT = load_model_from_checkpoint(BEST_CKPT_PATH, DEVICE)
BEST_MODEL.to(DEVICE).eval()
print(f"Loaded best_model.pt (epoch {BEST_CKPT['epoch']}, best_val_score={BEST_CKPT['best_val_score']:.4f}, "
      f"seed={BEST_CKPT['seed']}, model={BEST_CKPT['model_name']}, in_channels={BEST_CKPT.get('in_channels')})")

@torch.no_grad()
def predict_prob_tensor(model, pre, post, use_tta=False, use_amp=USE_AMP):
    """Return [B,1,H,W] probabilities. TTA averages 4 rotations × optional horizontal flip (8 transforms)."""
    if not use_tta:
        with autocast(device_type="cuda", enabled=use_amp and pre.device.type == "cuda"):
            return torch.sigmoid(model(pre, post))

    probs = []
    for k in range(4):
        for hflip in (False, True):
            pre_t = torch.rot90(pre, k, dims=(-2, -1))
            post_t = torch.rot90(post, k, dims=(-2, -1))
            if hflip:
                pre_t = torch.flip(pre_t, dims=(-1,)); post_t = torch.flip(post_t, dims=(-1,))
            with autocast(device_type="cuda", enabled=use_amp and pre.device.type == "cuda"):
                p = torch.sigmoid(model(pre_t, post_t))
            if hflip:
                p = torch.flip(p, dims=(-1,))
            p = torch.rot90(p, -k, dims=(-2, -1))
            probs.append(p)
    return torch.stack(probs, dim=0).mean(dim=0)

@torch.no_grad()
def collect_probs_targets(model, loader, device):
    all_probs, all_targets, all_valid, all_keys = [], [], [], []
    for batch in tqdm(loader, desc=f"Collecting predictions (TTA={INFERENCE_TTA})"):
        pre = batch["pre"].to(device); post = batch["post"].to(device)
        prob = predict_prob_tensor(model, pre, post, use_tta=INFERENCE_TTA).squeeze(1).cpu().numpy()
        target = batch["target"].numpy(); valid = batch["valid_mask"].numpy()
        for i in range(prob.shape[0]):
            all_probs.append(prob[i]); all_targets.append(target[i]); all_valid.append(valid[i]); all_keys.append(batch["key"][i])
    return all_probs, all_targets, all_valid, all_keys

Loaded best_model.pt (epoch 133, best_val_score=0.9035, seed=42, model=sam_cd, in_channels=1)


In [42]:
# Cell 25 — Validation-only threshold search (never uses test/external data)

val_probs, val_targets, val_valid, val_keys = collect_probs_targets(BEST_MODEL, VAL_LOADER, DEVICE)
thresholds = np.linspace(*THRESHOLD_SEARCH_RANGE, THRESHOLD_SEARCH_STEPS)
search_rows = []
for th in thresholds:
    acc = ConfusionAccumulator()
    for p, t, v in zip(val_probs, val_targets, val_valid):
        acc.update(p >= th, t.astype(bool), v.astype(bool))
    search_rows.append({"threshold": th, **acc.compute()})

threshold_search_df = pd.DataFrame(search_rows)
threshold_search_df.to_csv(os.path.join(DIRS["metrics"], "validation_threshold_search.csv"), index=False)
best_row = threshold_search_df.loc[threshold_search_df["iou"].idxmax()]
BEST_THRESHOLD = float(best_row["threshold"])
threshold_record = {"best_threshold": BEST_THRESHOLD, "criterion": "val_iou", "val_iou_at_best": float(best_row["iou"]),
                    "tta_used_for_validation": bool(INFERENCE_TTA)}
with open(os.path.join(DIRS["metrics"], "best_threshold.json"), "w") as f:
    json.dump(threshold_record, f, indent=2)
with open(os.path.join(DIRS["checkpoints"], "best_threshold.json"), "w") as f:
    json.dump(threshold_record, f, indent=2)

# FIX: persist the validation-selected threshold into the checkpoint actually used by predict_geotiff().
_ckpt_locked = load_checkpoint(BEST_CKPT_PATH, "cpu")
_ckpt_locked["threshold"] = BEST_THRESHOLD
_ckpt_locked["threshold_selection"] = threshold_record
torch.save(_ckpt_locked, BEST_CKPT_PATH)
BEST_CKPT = _ckpt_locked

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(threshold_search_df["threshold"], threshold_search_df["iou"], label="IoU")
ax.plot(threshold_search_df["threshold"], threshold_search_df["dice_f1"], label="Dice")
ax.axvline(BEST_THRESHOLD, linestyle="--", label=f"best={BEST_THRESHOLD:.3f}")
ax.set_xlabel("Threshold"); ax.set_ylabel("Score"); ax.legend(); ax.set_title("Validation Threshold Search")
fig.savefig(os.path.join(DIRS["figures"], "threshold_curve.png"), dpi=200, bbox_inches="tight")
plt.close(fig)

THRESHOLD = BEST_THRESHOLD
print(f"Locked THRESHOLD = {THRESHOLD:.3f} (validation IoU={best_row['iou']:.4f}) and persisted it into best_model.pt.")

Locked THRESHOLD = 0.850 (validation IoU=0.9114) and persisted it into best_model.pt.


In [43]:
# Cell 26 — Q1-level test-set evaluation (locked threshold, best checkpoint, TEST split only)

test_probs, test_targets, test_valid, test_keys = collect_probs_targets(BEST_MODEL, TEST_LOADER, DEVICE)

# Dataset-level (globally accumulated confusion counts)
global_acc = ConfusionAccumulator()
for p, t, v in zip(test_probs, test_targets, test_valid):
    global_acc.update(p >= THRESHOLD, t.astype(bool), v.astype(bool))
global_metrics = global_acc.compute()

all_true_flat = np.concatenate([t[v].astype(int) for t, v in zip(test_targets, test_valid)])
all_prob_flat = np.concatenate([p[v] for p, v in zip(test_probs, test_valid)])
all_pred_flat = (all_prob_flat >= THRESHOLD).astype(int)

global_metrics["kappa"] = cohen_kappa_score(all_true_flat, all_pred_flat)
global_metrics["mcc"] = matthews_corrcoef(all_true_flat, all_pred_flat)
global_metrics["pr_auc"] = average_precision_score(all_true_flat, all_prob_flat)
global_metrics["roc_auc"] = roc_auc_score(all_true_flat, all_prob_flat)
global_metrics["balanced_accuracy_sklearn"] = balanced_accuracy_score(all_true_flat, all_pred_flat)
global_metrics["threshold"] = THRESHOLD
global_metrics["n_test_images"] = len(test_keys)

test_metrics_summary = pd.DataFrame([global_metrics])
test_metrics_summary.to_csv(os.path.join(DIRS["metrics"], "test_metrics_summary.csv"), index=False)

# Per-image metrics
per_image_rows = []
for k, p, t, v in zip(test_keys, test_probs, test_targets, test_valid):
    m = per_image_metrics(p, t, v, THRESHOLD); m["key"] = k
    per_image_rows.append(m)
per_image_df = pd.DataFrame(per_image_rows)
per_image_df.to_csv(os.path.join(DIRS["metrics"], "test_metrics_per_image.csv"), index=False)

# Confusion matrix (dataset level)
cm = np.array([[global_metrics["tn"], global_metrics["fp"]], [global_metrics["fn"], global_metrics["tp"]]])
pd.DataFrame(cm, index=["true_0", "true_1"], columns=["pred_0", "pred_1"]).to_csv(
    os.path.join(DIRS["metrics"], "test_confusion_matrix.csv"))

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=["pred_non-water", "pred_water"], yticklabels=["true_non-water", "true_water"])
ax.set_title("Test Confusion Matrix (valid pixels, dataset-level)")
fig.savefig(os.path.join(DIRS["metrics"], "test_confusion_matrix.png"), dpi=200, bbox_inches="tight")
plt.close(fig)

print("TEST SET RESULTS (dataset-level, valid pixels only):")
for k in ["iou", "dice_f1", "precision", "recall", "specificity", "accuracy", "balanced_accuracy",
          "kappa", "mcc", "pr_auc", "roc_auc"]:
    print(f"  {k:>20s}: {global_metrics[k]:.4f}")

print("CAVEAT: S1GFloods filenames carry no event ID, so event-level split independence cannot be verified; "
      "benchmark test scores may be optimistic relative to external Gaibandha performance.")


TEST SET RESULTS (dataset-level, valid pixels only):
                   iou: 0.8406
               dice_f1: 0.9134
             precision: 0.8967
                recall: 0.9308
           specificity: 0.9740
              accuracy: 0.9656
     balanced_accuracy: 0.9524
                 kappa: 0.8920
                   mcc: 0.8922
                pr_auc: 0.9683
               roc_auc: 0.9912
CAVEAT: S1GFloods filenames carry no event ID, so event-level split independence cannot be verified; benchmark test scores may be optimistic relative to external Gaibandha performance.


In [44]:
# Cell 27 — OPTIONAL external reference-mask evaluation (run/re-run after Cell 35 external inference)
# Strictly compares on the existing raster grid; never resizes/reprojects the prediction or reference.

_expected_external_mask = globals().get("mask_path") or os.path.join(DIRS["inference_outputs"], "gaibandha", "flood_binary_mask.tif")
if EXTERNAL_REF_MASK is None:
    print("EXTERNAL_REF_MASK=None — optional external evaluation skipped.")
elif not os.path.exists(_expected_external_mask):
    print("External prediction mask not found yet. Run Cell 35, then re-run this cell:", _expected_external_mask)
elif not os.path.exists(EXTERNAL_REF_MASK):
    print("Configured EXTERNAL_REF_MASK does not exist:", EXTERNAL_REF_MASK)
else:
    with rasterio.open(_expected_external_mask) as pred_src, rasterio.open(EXTERNAL_REF_MASK) as ref_src:
        same_grid = (pred_src.width == ref_src.width and pred_src.height == ref_src.height and
                     pred_src.crs == ref_src.crs and pred_src.transform.almost_equals(ref_src.transform, precision=1e-9))
        if not same_grid:
            raise RuntimeError("External reference mask grid does not exactly match the prediction. No silent resampling is allowed.")
        pred = pred_src.read(1)
        ref = ref_src.read(1).astype(np.float32)
        valid = (pred != 255) & np.isfinite(ref)
        if ref_src.nodata is not None:
            valid &= ref != ref_src.nodata
        ref_pos = (ref > 0) if EXTERNAL_REF_POSITIVE_VALUE is None else (ref == EXTERNAL_REF_POSITIVE_VALUE)
        pred_pos = pred == 1
        acc = ConfusionAccumulator()
        acc.update(pred_pos, ref_pos, valid)
        ext_ref_metrics = acc.compute()
        pd.DataFrame([{k: ext_ref_metrics[k] for k in ["iou", "dice_f1", "precision", "recall", "tp", "fp", "fn", "tn"]}]).to_csv(
            os.path.join(DIRS["metrics"], "external_reference_metrics.csv"), index=False)
        print("External reference evaluation:")
        for k in ["iou", "dice_f1", "precision", "recall"]:
            print(f"  {k:>10s}: {ext_ref_metrics[k]:.4f}")


EXTERNAL_REF_MASK=None — optional external evaluation skipped.


In [45]:
# Cell 28 — Image-level bootstrap confidence intervals (test set)

def bootstrap_ci(per_image_df: pd.DataFrame, metrics: List[str], n_reps: int, ci: float, seed: int = 0):
    rng = np.random.RandomState(seed)
    n = len(per_image_df)
    rows = []
    boot_samples = {m: [] for m in metrics}
    for _ in range(n_reps):
        idx = rng.randint(0, n, size=n)  # bootstrap at the IMAGE level, not pixel level
        sample = per_image_df.iloc[idx]
        for m in metrics:
            boot_samples[m].append(np.nanmean(sample[m].values))
    alpha = (1 - ci) / 2
    for m in metrics:
        vals = np.array(boot_samples[m])
        rows.append({
            "metric": m,
            "point_estimate_mean_of_images": float(np.nanmean(per_image_df[m])),
            "bootstrap_mean": float(np.nanmean(vals)),
            f"ci_lower_{int(ci*100)}": float(np.nanpercentile(vals, alpha * 100)),
            f"ci_upper_{int(ci*100)}": float(np.nanpercentile(vals, (1 - alpha) * 100)),
            "n_bootstrap_reps": n_reps,
        })
    return pd.DataFrame(rows)

bootstrap_df = bootstrap_ci(per_image_df, ["iou", "dice_f1", "precision", "recall"],
                             n_reps=BOOTSTRAP_REPS, ci=BOOTSTRAP_CI, seed=SEED)
bootstrap_df.to_csv(os.path.join(DIRS["metrics"], "bootstrap_confidence_intervals.csv"), index=False)
bootstrap_df


,metric,point_estimate_mean_of_images,bootstrap_mean,ci_lower_95,ci_upper_95,n_bootstrap_reps
0,iou,0.837472,0.837816,0.826029,0.849895,1000
1,dice_f1,0.903447,0.903712,0.895038,0.912154,1000
2,precision,0.881810,0.881922,0.872588,0.891920,1000
3,recall,0.938137,0.938474,0.929664,0.946328,1000


In [46]:
# Cell 29 — Per-image analysis (distribution of scores, worst/best cases)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.histplot(per_image_df["iou"].dropna(), bins=20, ax=axes[0]); axes[0].set_title("Per-image IoU distribution")
sns.histplot(per_image_df["dice_f1"].dropna(), bins=20, ax=axes[1]); axes[1].set_title("Per-image Dice distribution")
fig.savefig(os.path.join(DIRS["figures"], "per_image_score_distributions.png"), dpi=200, bbox_inches="tight")
plt.close(fig)

worst5 = per_image_df.nsmallest(5, "iou")[["key", "iou", "dice_f1", "precision", "recall"]]
best5 = per_image_df.nlargest(5, "iou")[["key", "iou", "dice_f1", "precision", "recall"]]
print("Best 5 test images by IoU:\n", best5.to_string(index=False))
print("\nWorst 5 test images by IoU:\n", worst5.to_string(index=False))

worst5.to_csv(os.path.join(DIRS["metrics"], "worst5_test_images.csv"), index=False)
best5.to_csv(os.path.join(DIRS["metrics"], "best5_test_images.csv"), index=False)

Best 5 test images by IoU:
                 key      iou  dice_f1  precision   recall
test/image_4887.png 0.993473 0.996726   0.996067 0.997385
test/image_4862.png 0.990715 0.995336   0.999149 0.991552
test/image_5131.png 0.989403 0.994673   0.993968 0.995380
test/image_4863.png 0.984824 0.992354   0.988818 0.995916
test/image_4943.png 0.984305 0.992090   0.991469 0.992712

Worst 5 test images by IoU:
                 key      iou  dice_f1  precision   recall
test/image_5200.png 0.163108 0.280470   0.557392 0.187377
test/image_4925.png 0.253861 0.404927   0.303260 0.609137
test/image_5358.png 0.280288 0.437852   0.333488 0.637290
test/image_5208.png 0.282761 0.440863   0.504870 0.391259
test/image_5216.png 0.290566 0.450292   0.643678 0.346262


In [47]:
# Cell 30 — Publication-quality qualitative figures (random / best / worst)

def qualitative_panel(key, prob, target, valid, threshold, out_path):
    t = triplet_by_key[key]
    pre_gray = read_png_gray(t["pre"]); post_gray = read_png_gray(t["post"])
    diff_gray = post_gray.astype(np.float32) - pre_gray.astype(np.float32)

    pred_bin = prob >= threshold
    tp = pred_bin & (target == 1) & valid; fp = pred_bin & (target == 0) & valid
    fn = (~pred_bin) & (target == 1) & valid
    error_map = np.zeros((*target.shape, 3), dtype=np.float32)
    error_map[tp] = [0, 1, 0]; error_map[fp] = [1, 0, 0]; error_map[fn] = [0, 0, 1]

    fig, axes = plt.subplots(2, 4, figsize=(18, 9))
    panels = [
        (pre_gray, "PRE VV gray"), (post_gray, "POST VV gray"), (diff_gray, "POST-PRE gray"),
        (np.ma.masked_where(~valid, target), "Ground truth: new inundation"),
        (np.ma.masked_where(~valid, prob), "Probability map"),
        (np.ma.masked_where(~valid, pred_bin.astype(float)), "Binary prediction"),
        (error_map, "TP/FP/FN error map"), (valid.astype(float), "Valid mask"),
    ]
    for ax, (img, title) in zip(axes.flat, panels):
        if title == "POST-PRE gray":
            ax.imshow(img, cmap="RdBu", vmin=-80, vmax=80)
        elif title in {"PRE VV gray", "POST VV gray", "Valid mask"}:
            ax.imshow(img, cmap="gray")
        elif title == "Probability map":
            ax.imshow(img, cmap="viridis", vmin=0, vmax=1)
        elif title == "TP/FP/FN error map":
            ax.imshow(img)
        else:
            ax.imshow(img, cmap="Blues", vmin=0, vmax=1)
        ax.set_title(title, fontsize=11); ax.axis("off")
    fig.suptitle(f"Qualitative example: {key}", fontsize=13)
    fig.tight_layout(); fig.savefig(out_path, dpi=250, bbox_inches="tight"); plt.close(fig)

test_iou_by_key = dict(zip(per_image_df["key"], per_image_df["iou"]))
sorted_keys = sorted(test_iou_by_key, key=lambda k: test_iou_by_key[k])
selection = {"worst": sorted_keys[:3], "best": sorted_keys[-3:], "random": random.sample(test_keys, min(3, len(test_keys)))}
key_to_arrays = {k: (p, t, v) for k, p, t, v in zip(test_keys, test_probs, test_targets, test_valid)}
for group, keys in selection.items():
    for k in keys:
        p, t, v = key_to_arrays[k]
        qualitative_panel(k, p, t, v, THRESHOLD, os.path.join(DIRS["qualitative"], f"{group}_{k.replace('/', '_')}.png"))
print(f"Saved qualitative figures to {DIRS['qualitative']} for groups: {list(selection.keys())}")

Saved qualitative figures to /kaggle/working/samcd_s1gfloods/qualitative for groups: ['worst', 'best', 'random']


In [48]:
# Cell 30A — Save qualitative prediction figures as EPS for Adobe Illustrator

EPS_DIR = os.path.join(DIRS["qualitative"], "eps")
os.makedirs(EPS_DIR, exist_ok=True)

for group, keys in selection.items():
    for k in keys:
        p, t, v = key_to_arrays[k]

        eps_path = os.path.join(
            EPS_DIR,
            f"{group}_{k.replace('/', '_')}.eps"
        )

        qualitative_panel(
            k,
            p,
            t,
            v,
            THRESHOLD,
            eps_path
        )

print(f"EPS prediction figures saved to: {EPS_DIR}")
print(f"Total EPS files: {len(glob.glob(os.path.join(EPS_DIR, '*.eps')))}")


EPS prediction figures saved to: /kaggle/working/samcd_s1gfloods/qualitative/eps
Total EPS files: 9


In [49]:
# Cell 31 — Model complexity and runtime report

def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def model_size_mb(model):
    total_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    return total_bytes / (1024 ** 2)

total_params, trainable_params = count_params(BEST_MODEL)
size_mb = model_size_mb(BEST_MODEL)

try:
    from thop import profile
    dummy_pre = torch.randn(1, IN_CHANNELS, 256, 256).to(DEVICE)
    dummy_post = torch.randn(1, IN_CHANNELS, 256, 256).to(DEVICE)
    macs, _ = profile(BEST_MODEL, inputs=(dummy_pre, dummy_post), verbose=False)
    flops = macs * 2
except Exception as e:
    print(f"FLOPs/MACs profiling unavailable ({e}); leaving as NaN.")
    macs, flops = np.nan, np.nan

# Inference timing / throughput on a batch of validation tiles
BEST_MODEL.eval()
n_timing_batches = min(10, len(VAL_LOADER))
times = []
with torch.no_grad():
    it = iter(VAL_LOADER)
    for _ in range(n_timing_batches):
        batch = next(it)
        pre = batch["pre"].to(DEVICE); post = batch["post"].to(DEVICE)
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        t0 = time.time()
        _ = BEST_MODEL(pre, post)
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        times.append((time.time() - t0) / pre.shape[0])
mean_infer_time_per_image = float(np.mean(times)) if times else np.nan
peak_mem_mb = torch.cuda.max_memory_allocated(DEVICE) / (1024 ** 2) if DEVICE.type == "cuda" else np.nan

total_train_time = hist_df["epoch_time_sec"].sum() if "epoch_time_sec" in hist_df.columns else np.nan

complexity_df = pd.DataFrame([{
    "model_name": MODEL_NAME, "total_params": total_params, "trainable_params": trainable_params,
    "model_size_mb": size_mb, "macs": macs, "flops": flops,
    "mean_inference_time_sec_per_image": mean_infer_time_per_image,
    "throughput_images_per_sec": (1.0 / mean_infer_time_per_image) if mean_infer_time_per_image else np.nan,
    "peak_gpu_memory_mb": peak_mem_mb, "total_training_time_sec": total_train_time,
}])
complexity_df.to_csv(os.path.join(DIRS["metrics"], "complexity_runtime.csv"), index=False)
complexity_df


,model_name,total_params,trainable_params,model_size_mb,macs,flops,mean_inference_time_sec_per_image,throughput_images_per_sec,peak_gpu_memory_mb,total_training_time_sec
0,sam_cd,62025997,2591357,236.610401,4.328671e+10,8.657342e+10,0.047922,20.867156,4252.820312,18360.289376


In [50]:
# Cell 32 — Multi-seed training + aggregation (set RUN_SEEDS = [42,52,62,72,82] in Cell 4 for the final run)
# For the default development config (RUN_SEEDS=[42]) this reuses the run already completed above and simply
# reports it as a single-seed "multi-seed" table; set RUN_SEEDS to multiple values and RE-RUN Cells 22-31 in a
# loop (or re-run this notebook end-to-end per seed) to populate genuinely independent runs before aggregating.

def evaluate_checkpoint_on_test(ckpt_path, threshold):
    model, ckpt = load_model_from_checkpoint(ckpt_path, DEVICE)
    model.eval()
    probs, targets, valid, keys = collect_probs_targets(model, TEST_LOADER, DEVICE)
    acc = ConfusionAccumulator()
    for p, t, v in zip(probs, targets, valid):
        acc.update(p >= threshold, t.astype(bool), v.astype(bool))
    m = acc.compute()
    return m, ckpt.get("seed")

multi_seed_rows = []
for seed in RUN_SEEDS:
    # In the single-seed default case this is the run we already have; for RUN_SEEDS with >1 entries, re-run
    # train_one_seed(seed, CONFIG) for each seed BEFORE this cell, saving per-seed checkpoints to distinct
    # subfolders (e.g. checkpoints/seed_{seed}/) to avoid overwriting best_model.pt between seeds.
    m, ckpt_seed = evaluate_checkpoint_on_test(BEST_CKPT_PATH, THRESHOLD)
    m["seed"] = seed
    multi_seed_rows.append(m)

multi_seed_df = pd.DataFrame(multi_seed_rows)
summary = {"metric": [], "mean": [], "std": []}
for col in ["iou", "dice_f1", "precision", "recall", "accuracy"]:
    summary["metric"].append(col)
    summary["mean"].append(multi_seed_df[col].mean())
    summary["std"].append(multi_seed_df[col].std())
multi_seed_summary_df = pd.DataFrame(summary)
multi_seed_df.to_csv(os.path.join(DIRS["metrics"], "multi_seed_raw_results.csv"), index=False)
multi_seed_summary_df.to_csv(os.path.join(DIRS["metrics"], "multi_seed_summary.csv"), index=False)
print(f"NOTE: with RUN_SEEDS={RUN_SEEDS}, {'this reflects a single training run' if len(RUN_SEEDS)==1 else 'each row should come from an independently trained checkpoint'}.")
multi_seed_summary_df


NOTE: with RUN_SEEDS=[42], this reflects a single training run.


,metric,mean,std
0,iou,0.840645,NaN
1,dice_f1,0.913425,NaN
2,precision,0.896680,NaN
3,recall,0.930806,NaN
4,accuracy,0.965617,NaN


In [51]:
# Cell 33 — Journal-ready results table

paper_row = {
    "Model": MODEL_NAME,
    "Input": f"Bi-temporal S1GFloods VV PRE+POST ({INPUT_MODE}, {IN_CHANNELS} ch/date)",
    "Loss": LOSS_NAME,
    "Parameters": total_params,
    "Trainable Parameters": trainable_params,
    "IoU": global_metrics["iou"],
    "Dice/F1": global_metrics["dice_f1"],
    "Precision": global_metrics["precision"],
    "Recall": global_metrics["recall"],
    "Specificity": global_metrics["specificity"],
    "Accuracy": global_metrics["accuracy"],
    "Balanced Accuracy": global_metrics["balanced_accuracy"],
    "Kappa": global_metrics["kappa"],
    "MCC": global_metrics["mcc"],
    "PR-AUC": global_metrics["pr_auc"],
    "Inference Time (s/image)": mean_infer_time_per_image,
}
if len(RUN_SEEDS) > 1:
    for m in ["iou", "dice_f1", "precision", "recall"]:
        row = multi_seed_summary_df[multi_seed_summary_df["metric"] == m].iloc[0]
        paper_row[f"{m}_mean_std"] = f"{row['mean']:.4f} ± {row['std']:.4f}"

paper_results_table = pd.DataFrame([paper_row])
paper_results_table.to_csv(os.path.join(DIRS["metrics"], "paper_results_table.csv"), index=False)
try:
    paper_results_table.to_excel(os.path.join(DIRS["metrics"], "paper_results_table.xlsx"), index=False)
except Exception as e:
    print(f"XLSX export skipped ({e}); CSV is available.")
paper_results_table


,Model,Input,Loss,Parameters,Trainable Parameters,IoU,Dice/F1,Precision,Recall,Specificity,Accuracy,Balanced Accuracy,Kappa,MCC,PR-AUC,Inference Time (s/image)
0,sam_cd,"Bi-temporal S1GFloods VV PRE+POST (1ch, 1 ch/d...",bce_dice,62025997,2591357,0.840645,0.913425,0.89668,0.930806,0.974042,0.965617,0.952424,0.891983,0.892225,0.968284,0.047922


In [52]:
# Cell 33B — OPTIONAL side-by-side comparison with the Siamese-UNet run (skipped if BASELINE_RESULTS_DIR is None)
# Both runs use the identical deterministic split, so per-image metrics are paired by test-image key.
from scipy.stats import wilcoxon

def _find_baseline_file(base_dir, name):
    for p in [os.path.join(base_dir, "metrics", name), os.path.join(base_dir, name)]:
        if os.path.exists(p):
            return p
    return None

if BASELINE_RESULTS_DIR is None:
    print("BASELINE_RESULTS_DIR=None — comparison with the Siamese-UNet run skipped. "
          "Set it in Cell 4 to the baseline output folder to enable it.")
else:
    base_summary_p = _find_baseline_file(BASELINE_RESULTS_DIR, "test_metrics_summary.csv")
    base_img_p = _find_baseline_file(BASELINE_RESULTS_DIR, "test_metrics_per_image.csv")
    if base_summary_p is None or base_img_p is None:
        print("Could not find test_metrics_summary.csv / test_metrics_per_image.csv under", BASELINE_RESULTS_DIR)
    else:
        base_sum = pd.read_csv(base_summary_p).iloc[0]
        base_img = pd.read_csv(base_img_p)
        base_cx_p = _find_baseline_file(BASELINE_RESULTS_DIR, "complexity_runtime.csv")
        base_cx = pd.read_csv(base_cx_p).iloc[0] if base_cx_p else None

        rows = []
        for m in ["iou", "dice_f1", "precision", "recall", "specificity", "accuracy", "balanced_accuracy",
                  "kappa", "mcc", "pr_auc", "roc_auc"]:
            if m in base_sum.index and m in global_metrics:
                rows.append({"metric (dataset-level, test)": m, "baseline (Siamese-UNet)": float(base_sum[m]),
                             f"{MODEL_NAME}": float(global_metrics[m]),
                             "difference (model - baseline)": float(global_metrics[m]) - float(base_sum[m])})
        compare_df = pd.DataFrame(rows)
        compare_df.to_csv(os.path.join(DIRS["metrics"], "benchmark_comparison_global.csv"), index=False)
        print(compare_df.to_string(index=False))

        # Paired image-level analysis on test images present in both runs.
        merged = per_image_df.merge(base_img, on="key", suffixes=("_model", "_base"))
        print(f"\nPaired test images: {len(merged)} (model: {len(per_image_df)}, baseline: {len(base_img)})")
        if len(merged) != len(per_image_df) or len(merged) != len(base_img):
            print("WARNING: test sets differ between runs; check that both used the same DATA_ROOT / split.")
        paired_rows = []
        rng = np.random.RandomState(SEED)
        for m in ["iou", "dice_f1", "precision", "recall"]:
            a = merged[f"{m}_model"].values.astype(float); b = merged[f"{m}_base"].values.astype(float)
            ok_ = np.isfinite(a) & np.isfinite(b); a, b = a[ok_], b[ok_]
            d = a - b
            boots = np.array([d[rng.randint(0, len(d), len(d))].mean() for _ in range(BOOTSTRAP_REPS)])
            lo_, hi_ = np.percentile(boots, [(1 - BOOTSTRAP_CI) / 2 * 100, (1 + BOOTSTRAP_CI) / 2 * 100])
            try:
                p_val = float(wilcoxon(a, b).pvalue) if np.any(d != 0) else 1.0
            except Exception:
                p_val = float("nan")
            paired_rows.append({"metric (mean over images)": m, "baseline_mean": float(b.mean()), "model_mean": float(a.mean()),
                                "mean_diff": float(d.mean()), f"diff_ci_lower_{int(BOOTSTRAP_CI*100)}": float(lo_),
                                f"diff_ci_upper_{int(BOOTSTRAP_CI*100)}": float(hi_), "wilcoxon_p": p_val,
                                "n_images": int(len(d)), "model_wins_frac": float((d > 0).mean())})
        paired_df = pd.DataFrame(paired_rows)
        paired_df.to_csv(os.path.join(DIRS["metrics"], "benchmark_comparison_paired.csv"), index=False)
        print("\nPaired bootstrap / Wilcoxon (difference = model - baseline):")
        print(paired_df.to_string(index=False))

        if base_cx is not None:
            cx = pd.DataFrame([{"model": "baseline (Siamese-UNet)", "total_params": base_cx.get("total_params"),
                                "trainable_params": base_cx.get("trainable_params"),
                                "mean_inference_time_sec_per_image": base_cx.get("mean_inference_time_sec_per_image"),
                                "total_training_time_sec": base_cx.get("total_training_time_sec")},
                               {"model": MODEL_NAME, "total_params": total_params, "trainable_params": trainable_params,
                                "mean_inference_time_sec_per_image": mean_infer_time_per_image,
                                "total_training_time_sec": total_train_time}])
            cx.to_csv(os.path.join(DIRS["metrics"], "benchmark_comparison_complexity.csv"), index=False)
            print("\nComplexity:"); print(cx.to_string(index=False))


BASELINE_RESULTS_DIR=None — comparison with the Siamese-UNet run skipped. Set it in Cell 4 to the baseline output folder to enable it.


In [53]:
# Cell 34 — Real Sentinel-1 GeoTIFF inference: VV name lookup, checkpoint preprocessing, safe tiling

def hann_window_2d(size: int) -> np.ndarray:
    w1 = np.hanning(size).astype(np.float32)
    w1[w1 == 0] = 1e-3
    return np.outer(w1, w1).astype(np.float32)


def get_vv_band_index(src, path_label="GeoTIFF") -> int:
    descriptions = list(src.descriptions or [])
    if not descriptions or all(d is None or str(d).strip() == "" for d in descriptions):
        print(f"WARNING: {path_label} has no band descriptions; UNVERIFIED fallback assumes band 1 is VV.")
        return 1
    names = [str(d).strip().upper() if d is not None else "" for d in descriptions]
    if "VV" not in names:
        raise RuntimeError(f"{path_label} has band descriptions {descriptions} but no band named 'VV'. Refusing to guess.")
    return names.index("VV") + 1


def _bounds_close(a, b, atol=1e-6):
    return np.allclose([a.left, a.bottom, a.right, a.top], [b.left, b.bottom, b.right, b.top], rtol=0, atol=atol)


def check_pre_post_compatibility(pre_path, post_path):
    with rasterio.open(pre_path) as p, rasterio.open(post_path) as q:
        mismatches = {}
        if p.width != q.width: mismatches["width"] = (p.width, q.width)
        if p.height != q.height: mismatches["height"] = (p.height, q.height)
        if p.crs != q.crs: mismatches["crs"] = (str(p.crs), str(q.crs))
        if not p.transform.almost_equals(q.transform, precision=1e-9): mismatches["transform"] = (p.transform, q.transform)
        if not np.allclose(p.res, q.res, rtol=0, atol=1e-9): mismatches["res"] = (p.res, q.res)
        if not _bounds_close(p.bounds, q.bounds): mismatches["bounds"] = (p.bounds, q.bounds)
        if mismatches:
            msg = "PRE/POST GeoTIFF compatibility check FAILED:\n" + "\n".join(f"  {k}: {v}" for k, v in mismatches.items())
            if AUTO_ALIGN:
                raise RuntimeError(msg + "\nAUTO_ALIGN=True does not trigger silent resampling; align explicitly outside this function.")
            raise RuntimeError(msg)

        pre_vv = get_vv_band_index(p, "PRE GeoTIFF")
        post_vv = get_vv_band_index(q, "POST GeoTIFF")
        if p.crs is not None and p.crs.is_geographic:
            print("WARNING: CRS is geographic (degrees). Pixel resolution is angular, not 10 m; area uses latitude-dependent row areas.")
        elif not (abs(abs(p.res[0]) - 10.0) < 0.5 and abs(abs(p.res[1]) - 10.0) < 0.5):
            print(f"WARNING: projected pixel resolution {p.res} differs from expected ~10 m.")
        print("PRE/POST compatibility PASSED: width/height/CRS/transform/resolution/bounds match; VV is available in both.")
        return {
            "width": p.width, "height": p.height, "crs": p.crs, "transform": p.transform, "res": p.res,
            "pre_vv_index": pre_vv, "post_vv_index": post_vv,
        }


def _external_calibration_db(vv_db: np.ndarray, preproc: S1GFloodsPreprocessor, mode: str) -> np.ndarray:
    """Optional identical PRE/POST two-point calibration. OFF by default; when on, output is approximate."""
    if mode == "none":
        return vv_db.astype(np.float32, copy=False)
    if mode != "two_point":
        raise ValueError("EXTERNAL_CALIBRATION must be 'none' or 'two_point'.")
    if CALIBRATION_STABLE_LAND_DB is None or CALIBRATION_RIVER_DB is None:
        raise ValueError("two_point calibration requires CALIBRATION_STABLE_LAND_DB and CALIBRATION_RIVER_DB.")
    x_water = float(CALIBRATION_RIVER_DB); x_land = float(CALIBRATION_STABLE_LAND_DB)
    if x_land == x_water:
        raise ValueError("Calibration stable-land and river dB anchors must differ.")
    y_water = preproc.lo + (CALIBRATION_TARGET_WATER_GRAY / 255.0) * (preproc.hi - preproc.lo)
    y_land = preproc.lo + (CALIBRATION_TARGET_LAND_GRAY / 255.0) * (preproc.hi - preproc.lo)
    a = (y_land - y_water) / (x_land - x_water)
    b = y_water - a * x_water
    out = a * vv_db.astype(np.float32) + b
    out[~np.isfinite(vv_db)] = np.nan
    return out.astype(np.float32)


def _safe_pad_2d(arr: np.ndarray, tile_size: int, constant_value: float = 0.0) -> np.ndarray:
    """Pad bottom/right to tile_size. Reflect only when requested pad is smaller than that dimension."""
    out = np.asarray(arr)
    pad_h = tile_size - out.shape[0]
    pad_w = tile_size - out.shape[1]
    if pad_h < 0 or pad_w < 0:
        raise ValueError(f"Array {out.shape} exceeds tile_size={tile_size}")
    if pad_h:
        mode = "reflect" if out.shape[0] > 1 and pad_h < out.shape[0] else "constant"
        kwargs = {} if mode == "reflect" else {"constant_values": constant_value}
        out = np.pad(out, ((0, pad_h), (0, 0)), mode=mode, **kwargs)
    if pad_w:
        mode = "reflect" if out.shape[1] > 1 and pad_w < out.shape[1] else "constant"
        kwargs = {} if mode == "reflect" else {"constant_values": constant_value}
        out = np.pad(out, ((0, 0), (0, pad_w)), mode=mode, **kwargs)
    return out


def _window_grid(width, height, block=2048):
    for y in range(0, height, block):
        for x in range(0, width, block):
            yield Window(x, y, min(block, width-x), min(block, height-y))


def gray_histogram_from_raster(src, vv_idx: int, preproc: S1GFloodsPreprocessor,
                               calibration_mode="none", scope_window: Optional[Window] = None) -> np.ndarray:
    """Streaming 256-bin histogram in the mapped gray domain; invalid pixels are excluded."""
    hist = np.zeros(256, dtype=np.int64)
    if scope_window is None:
        windows = _window_grid(src.width, src.height)
    else:
        x0, y0, w0, h0 = map(int, [scope_window.col_off, scope_window.row_off, scope_window.width, scope_window.height])
        windows = (Window(x0+x, y0+y, min(2048, w0-x), min(2048, h0-y))
                   for y in range(0, h0, 2048) for x in range(0, w0, 2048))
    for win in windows:
        raw = src.read(vv_idx, window=win).astype(np.float32)
        raw = _external_calibration_db(raw, preproc, calibration_mode)
        x, valid = preproc.from_db(raw, fill_gray=0.0)
        if valid.any():
            g = np.clip(np.rint(x[valid] * 255.0), 0, 255).astype(np.uint8)
            hist += np.bincount(g, minlength=256)
    return hist


def histogram_median_gray(hist: np.ndarray) -> float:
    total = int(hist.sum())
    if total == 0:
        return 0.0
    return float(np.searchsorted(np.cumsum(hist), (total + 1) / 2))


def _resolve_nodata_fill(src, vv_idx, preproc, nodata_fill, calibration_mode="none", scope_window=None):
    if nodata_fill == "zero":
        return 0.0
    if nodata_fill != "median":
        raise ValueError("NODATA_FILL must be 'median' or 'zero'.")
    hist = gray_histogram_from_raster(src, vv_idx, preproc, calibration_mode, scope_window=scope_window)
    return histogram_median_gray(hist)


def _resolve_locked_threshold(ckpt, checkpoint_path):
    sidecar = os.path.join(os.path.dirname(checkpoint_path), "best_threshold.json")
    if os.path.exists(sidecar):
        try:
            with open(sidecar) as f:
                rec = json.load(f)
            return float(rec["best_threshold"]), "sidecar+checkpoint"
        except Exception as e:
            print("WARNING: could not read threshold sidecar; falling back to checkpoint:", e)
    return float(ckpt.get("threshold", THRESHOLD)), "checkpoint"


def _run_tiled_inference(model, pre_src, post_src, pre_vv_idx, post_vv_idx, preproc,
                         base_window: Window, tile_size: int, overlap: int, device,
                         nodata_fill: str, tta: bool, calibration_mode: str,
                         fill_scope_window: Optional[Window] = None):
    width = int(base_window.width); height = int(base_window.height)
    base_x = int(base_window.col_off); base_y = int(base_window.row_off)
    if tile_size <= overlap:
        raise ValueError("tile_size must be larger than overlap.")
    stride = tile_size - overlap
    weight = hann_window_2d(tile_size)

    # Primary full-scene inference uses scene-level medians. Diagnostic ROI inference may pass an ROI scope explicitly.
    pre_fill = _resolve_nodata_fill(pre_src, pre_vv_idx, preproc, nodata_fill, calibration_mode, scope_window=fill_scope_window)
    post_fill = _resolve_nodata_fill(post_src, post_vv_idx, preproc, nodata_fill, calibration_mode, scope_window=fill_scope_window)
    print(f"NoData fill gray: PRE={pre_fill:.1f}, POST={post_fill:.1f} ({nodata_fill})")

    prob_accum = np.zeros((height, width), dtype=np.float32)
    weight_accum = np.zeros((height, width), dtype=np.float32)
    valid_accum = np.zeros((height, width), dtype=bool)

    def tile_starts(length):
        if length <= tile_size:
            return [0]
        starts = list(range(0, length - tile_size + 1, stride))
        last = length - tile_size
        if starts[-1] != last:
            starts.append(last)
        return starts

    y_positions = tile_starts(height); x_positions = tile_starts(width)
    skipped = 0; total_tiles = len(y_positions) * len(x_positions)

    model.eval()
    with torch.no_grad():
        for y0 in tqdm(y_positions, desc="Inference rows"):
            for x0 in x_positions:
                h = min(tile_size, height-y0); w = min(tile_size, width-x0)
                win = Window(base_x+x0, base_y+y0, w, h)
                pre_db = pre_src.read(pre_vv_idx, window=win).astype(np.float32)
                post_db = post_src.read(post_vv_idx, window=win).astype(np.float32)
                pre_db = _external_calibration_db(pre_db, preproc, calibration_mode)
                post_db = _external_calibration_db(post_db, preproc, calibration_mode)

                pre01, pre_valid = preproc.from_db(pre_db, fill_gray=pre_fill)
                post01, post_valid = preproc.from_db(post_db, fill_gray=post_fill)
                valid = pre_valid & post_valid
                if not valid.any():
                    skipped += 1
                    continue

                pre01 = _safe_pad_2d(pre01, tile_size, constant_value=pre_fill/255.0)
                post01 = _safe_pad_2d(post01, tile_size, constant_value=post_fill/255.0)
                pre_in = preproc.expand_channels(pre01); post_in = preproc.expand_channels(post01)
                assert pre_in.shape == (preproc.in_channels, tile_size, tile_size)
                assert post_in.shape == (preproc.in_channels, tile_size, tile_size)
                pre_t = torch.from_numpy(pre_in).unsqueeze(0).float().to(device)
                post_t = torch.from_numpy(post_in).unsqueeze(0).float().to(device)
                prob_tile = predict_prob_tensor(model, pre_t, post_t, use_tta=tta).squeeze().cpu().numpy()

                w_use = weight[:h, :w]
                prob_accum[y0:y0+h, x0:x0+w] += prob_tile[:h, :w] * w_use
                weight_accum[y0:y0+h, x0:x0+w] += w_use
                valid_accum[y0:y0+h, x0:x0+w] |= valid

    final_prob = np.divide(prob_accum, weight_accum, out=np.zeros_like(prob_accum), where=weight_accum > 0)
    final_prob[~valid_accum] = np.nan
    return final_prob, valid_accum, {"n_tiles_total": total_tiles, "n_tiles_skipped_fully_invalid": skipped,
                                     "pre_fill_gray": pre_fill, "post_fill_gray": post_fill}


def _projected_linear_unit_to_m(crs) -> float:
    try:
        factor = crs.linear_units_factor
        if isinstance(factor, (tuple, list)):
            return float(factor[-1])
        return float(factor)
    except Exception:
        return 1.0


def area_statistics(binary_mask: np.ndarray, valid_mask: np.ndarray, transform, crs) -> Dict[str, Any]:
    flood = (binary_mask == 1) & valid_mask
    valid = valid_mask.astype(bool)
    if crs is not None and crs.is_geographic:
        if abs(transform.b) > 1e-12 or abs(transform.d) > 1e-12:
            print("WARNING: geographic raster has rotation/shear; row-area approximation assumes a north-up grid.")
        R = 6371008.8
        dlon = math.radians(abs(transform.a))
        flood_area = 0.0; valid_area = 0.0
        row_areas = []
        for r in range(binary_mask.shape[0]):
            lat_top = (transform * (0, r))[1]
            lat_bottom = (transform * (0, r+1))[1]
            px_area = R * R * dlon * abs(math.sin(math.radians(lat_top)) - math.sin(math.radians(lat_bottom)))
            row_areas.append(px_area)
            flood_area += int(flood[r].sum()) * px_area
            valid_area += int(valid[r].sum()) * px_area
        method = "latitude-dependent spherical row area"
        pixel_area = float(np.mean(row_areas)) if row_areas else np.nan
    else:
        unit_to_m = _projected_linear_unit_to_m(crs) if crs is not None else 1.0
        pixel_area = abs(transform.a * transform.e - transform.b * transform.d) * (unit_to_m ** 2)
        flood_area = float(flood.sum()) * pixel_area
        valid_area = float(valid.sum()) * pixel_area
        method = "projected affine determinant"
    return {
        "area_method": method,
        "representative_pixel_area_m2": float(pixel_area),
        "n_flood_pixels": int(flood.sum()), "n_valid_pixels": int(valid.sum()),
        "flood_area_m2": float(flood_area), "flood_area_km2": float(flood_area / 1e6),
        "valid_area_km2": float(valid_area / 1e6),
        "flood_percent_of_valid_area": float(100.0 * flood_area / valid_area) if valid_area > 0 else np.nan,
    }


def predict_geotiff(pre_tif, post_tif, checkpoint_path, output_dir,
                     tile_size=INFERENCE_TILE_SIZE, overlap=INFERENCE_OVERLAP, device=None,
                     tta=INFERENCE_TTA, nodata_fill=NODATA_FILL, calibration_mode=EXTERNAL_CALIBRATION,
                     preprocessor_override: Optional[S1GFloodsPreprocessor] = None,
                     roi_window: Optional[Window] = None, diagnostic_fill_from_roi: bool = False):
    os.makedirs(output_dir, exist_ok=True)
    device = device or DEVICE
    model, ckpt = load_model_from_checkpoint(checkpoint_path, device)
    model.eval()
    preproc = preprocessor_override or S1GFloodsPreprocessor.from_dict(ckpt["preprocessing"])
    threshold, threshold_source = _resolve_locked_threshold(ckpt, checkpoint_path)
    print(f"Inference threshold = {threshold:.4f} ({threshold_source})")
    print("Preprocessing loaded from checkpoint" if preprocessor_override is None else "DIAGNOSTIC: preprocessing override supplied")
    print("dB→gray lo/hi used:", preproc.lo, preproc.hi)

    compat = check_pre_post_compatibility(pre_tif, post_tif)
    with rasterio.open(pre_tif) as pre_src, rasterio.open(post_tif) as post_src:
        if roi_window is None:
            base_window = Window(0, 0, pre_src.width, pre_src.height)
        else:
            base_window = Window(*map(int, [roi_window.col_off, roi_window.row_off, roi_window.width, roi_window.height]))
            if (base_window.col_off < 0 or base_window.row_off < 0 or
                base_window.col_off + base_window.width > pre_src.width or
                base_window.row_off + base_window.height > pre_src.height):
                raise ValueError(f"ROI window {base_window} falls outside raster bounds {pre_src.width}×{pre_src.height}")

        fill_scope = base_window if diagnostic_fill_from_roi else None
        final_prob, valid, tile_meta = _run_tiled_inference(
            model, pre_src, post_src, compat["pre_vv_index"], compat["post_vv_index"], preproc,
            base_window, tile_size, overlap, device, nodata_fill, tta, calibration_mode,
            fill_scope_window=fill_scope,
        )
        transform = pre_src.window_transform(base_window)
        crs = pre_src.crs

    final_binary = np.where(valid, (final_prob >= threshold).astype(np.uint8), 255).astype(np.uint8)
    calibrated = calibration_mode != "none"
    prefix = "calibrated_approximate_" if calibrated else ""
    out_meta = dict(driver="GTiff", height=final_prob.shape[0], width=final_prob.shape[1], count=1,
                    crs=crs, transform=transform, compress="lzw")

    prob_path = os.path.join(output_dir, f"{prefix}flood_probability.tif")
    with rasterio.open(prob_path, "w", dtype="float32", nodata=np.nan, **out_meta) as dst:
        dst.write(final_prob.astype(np.float32), 1)
    mask_path = os.path.join(output_dir, f"{prefix}flood_binary_mask.tif")
    with rasterio.open(mask_path, "w", dtype="uint8", nodata=255, **out_meta) as dst:
        dst.write(final_binary, 1)
    validity_path = os.path.join(output_dir, f"{prefix}validity_mask.tif")
    with rasterio.open(validity_path, "w", dtype="uint8", nodata=None, **out_meta) as dst:
        dst.write(valid.astype(np.uint8), 1)

    fig, ax = plt.subplots(figsize=(10, 10 * final_prob.shape[0] / final_prob.shape[1]))
    im = ax.imshow(np.ma.masked_invalid(final_prob), cmap="Blues", vmin=0, vmax=1)
    title = "Flood probability quicklook"
    if calibrated: title += " — CALIBRATED / APPROXIMATE"
    ax.set_title(title); ax.axis("off"); plt.colorbar(im, ax=ax, fraction=0.03)
    quicklook_path = os.path.join(output_dir, f"{prefix}quicklook.png")
    fig.savefig(quicklook_path, dpi=200, bbox_inches="tight"); plt.close(fig)

    area = area_statistics(final_binary, valid, transform, crs)
    meta_out = {
        "pre_tif": pre_tif, "post_tif": post_tif, "checkpoint": checkpoint_path,
        "model_name": ckpt.get("model_name"), "checkpoint_epoch": ckpt.get("epoch"),
        "threshold_used": threshold, "threshold_source": threshold_source,
        "preprocessing": preproc.to_dict(), "png_db_lo_used": preproc.lo, "png_db_hi_used": preproc.hi,
        "preprocessor_source": "diagnostic_override" if preprocessor_override is not None else "checkpoint",
        "external_input_is_composite_UNVERIFIED": bool(EXTERNAL_INPUT_IS_COMPOSITE),
        "nodata_fill": nodata_fill, "tile_size": tile_size, "overlap": overlap, "tta": bool(tta),
        "calibration_mode": calibration_mode,
        "calibrated_approximate": calibrated,
        "roi_window": [int(base_window.col_off), int(base_window.row_off), int(base_window.width), int(base_window.height)] if roi_window is not None else None,
        "valid_pixel_fraction": float(valid.mean()), "crs": str(crs), "transform": tuple(transform),
        **tile_meta, **area, "timestamp_utc": pd.Timestamp.utcnow().isoformat(),
    }
    with open(os.path.join(output_dir, f"{prefix}external_inference_metadata.json"), "w") as f:
        json.dump(meta_out, f, indent=2, default=str)
    print(f"Flood area: {area['flood_area_km2']:.4f} km² ({area['flood_percent_of_valid_area']:.3f}% of valid area)")
    return prob_path, mask_path, validity_path, meta_out

print("predict_geotiff() ready. Primary inference restores preprocessing and threshold from best_model.pt.")


predict_geotiff() ready. Primary inference restores preprocessing and threshold from best_model.pt.


In [54]:
# Cell 35 — Example external inference call (Gaibandha)
# The external raster is never resized. Real VV dB is mapped into the checkpoint's S1GFloods gray domain.

if os.path.exists(EXTERNAL_PRE_TIF) and os.path.exists(EXTERNAL_POST_TIF):
    ext_out_dir = os.path.join(DIRS["inference_outputs"], "gaibandha")
    prob_path, mask_path, validity_path, ext_meta = predict_geotiff(
        EXTERNAL_PRE_TIF, EXTERNAL_POST_TIF, BEST_CKPT_PATH, ext_out_dir,
        tile_size=INFERENCE_TILE_SIZE, overlap=INFERENCE_OVERLAP, device=DEVICE,
        tta=INFERENCE_TTA, nodata_fill=NODATA_FILL, calibration_mode=EXTERNAL_CALIBRATION,
    )
    print("External inference complete:")
    print(" ", prob_path); print(" ", mask_path); print(" ", validity_path)
    print("Metadata:", json.dumps(ext_meta, indent=2, default=str))
    if EXTERNAL_CALIBRATION != "none":
        print("WARNING: these outputs are explicitly CALIBRATED / APPROXIMATE, not raw checkpoint-domain outputs.")
else:
    print("External GeoTIFFs not found. Update EXTERNAL_PRE_TIF / EXTERNAL_POST_TIF in Cell 4 and re-run this cell.")
    ext_out_dir, ext_meta = None, None
    prob_path = mask_path = validity_path = None

Inference threshold = 0.8500 (sidecar+checkpoint)
Preprocessing loaded from checkpoint
dB→gray lo/hi used: -30.0 0.0
PRE/POST compatibility PASSED: width/height/CRS/transform/resolution/bounds match; VV is available in both.
NoData fill gray: PRE=181.0, POST=173.0 (median)


Inference rows:   0%|          | 0/52 [00:00<?, ?it/s]

Flood area: 302.1776 km² (13.903% of valid area)
External inference complete:
  /kaggle/working/samcd_s1gfloods/inference_outputs/gaibandha/flood_probability.tif
  /kaggle/working/samcd_s1gfloods/inference_outputs/gaibandha/flood_binary_mask.tif
  /kaggle/working/samcd_s1gfloods/inference_outputs/gaibandha/validity_mask.tif
Metadata: {
  "pre_tif": "/kaggle/input/datasets/ashrif411/gaibandha/SPARRSO/Gaibandha_S1_pre_June 8_20 2024.tif",
  "post_tif": "/kaggle/input/datasets/ashrif411/gaibandha/SPARRSO/Gaibandha_S1_post_June 25_July 10_2024.tif",
  "checkpoint": "/kaggle/working/samcd_s1gfloods/checkpoints/best_model.pt",
  "model_name": "sam_cd",
  "checkpoint_epoch": 133,
  "threshold_used": 0.85,
  "threshold_source": "sidecar+checkpoint",
  "preprocessing": {
    "lo": -30.0,
    "hi": 0.0,
    "quantize": true,
    "in_channels": 1,
    "input_mode": "1ch",
    "png_scale": 255.0,
    "expected_units": "dB",
    "band_name": "VV",
    "expected_resolution_m": 10
  },
  "png_db_lo_u

In [55]:
# Cell 35B — MANDATORY GeoTIFF/PNG preprocessing parity acceptance test
# Converts 3 S1GFloods test PNG pairs back to dB with the checkpoint lo/hi, writes synthetic VV/VH GeoTIFFs,
# then requires deployment probabilities to match the PNG dataset path within PARITY_TOLERANCE without TTA.

PARITY_TEST_PASSED = False
parity_dir = os.path.join(DIRS["inference_outputs"], "geotiff_png_parity")
os.makedirs(parity_dir, exist_ok=True)
parity_model, parity_ckpt = load_model_from_checkpoint(BEST_CKPT_PATH, DEVICE)
parity_preproc = S1GFloodsPreprocessor.from_dict(parity_ckpt["preprocessing"])
parity_rows = []

for j, t in enumerate(SPLITS["test"][:PARITY_TEST_PAIRS]):
    pair_dir = os.path.join(parity_dir, f"pair_{j}"); os.makedirs(pair_dir, exist_ok=True)
    a_gray = read_png_gray(t["pre"]); b_gray = read_png_gray(t["post"])
    # Exact inverse of the assumed linear mapping at the PNG quantization grid.
    a_db = parity_preproc.lo + (a_gray.astype(np.float32) / 255.0) * (parity_preproc.hi - parity_preproc.lo)
    b_db = parity_preproc.lo + (b_gray.astype(np.float32) / 255.0) * (parity_preproc.hi - parity_preproc.lo)
    transform = from_origin(500000 + j*3000, 2600000, 10, 10)
    meta = dict(driver="GTiff", height=a_gray.shape[0], width=a_gray.shape[1], count=2,
                dtype="float32", crs="EPSG:32646", transform=transform)
    pre_tif = os.path.join(pair_dir, "pre_synthetic.tif"); post_tif = os.path.join(pair_dir, "post_synthetic.tif")
    for path, vv in [(pre_tif, a_db), (post_tif, b_db)]:
        with rasterio.open(path, "w", **meta) as dst:
            dst.write(vv.astype(np.float32), 1); dst.write(np.zeros_like(vv, dtype=np.float32), 2)
            dst.set_band_description(1, "VV"); dst.set_band_description(2, "VH")

    # PNG dataset path, no augmentation.
    ds = S1GFloodsChangeDataset([t], parity_preproc, spatial_aug=None, radiometric_aug=None)
    item = ds[0]
    with torch.no_grad():
        p_png = predict_prob_tensor(parity_model, item["pre"].unsqueeze(0).to(DEVICE),
                                    item["post"].unsqueeze(0).to(DEVICE), use_tta=False).squeeze().cpu().numpy()

    p_path, _, _, _ = predict_geotiff(
        pre_tif, post_tif, BEST_CKPT_PATH, os.path.join(pair_dir, "geotiff_inference"),
        tile_size=INFERENCE_TILE_SIZE, overlap=INFERENCE_OVERLAP, device=DEVICE,
        tta=False, nodata_fill="zero", calibration_mode="none",
    )
    with rasterio.open(p_path) as srcp:
        p_tif = srcp.read(1)
    max_abs_diff = float(np.nanmax(np.abs(p_png - p_tif)))
    parity_rows.append({"key": t["key"], "max_abs_probability_difference": max_abs_diff})
    assert max_abs_diff < PARITY_TOLERANCE, (
        f"GeoTIFF/PNG parity FAILED for {t['key']}: max abs diff {max_abs_diff:.6f} >= {PARITY_TOLERANCE}. "
        "Training and inference preprocessing have diverged."
    )

parity_df = pd.DataFrame(parity_rows)
parity_df.to_csv(os.path.join(parity_dir, "parity_test_results.csv"), index=False)
PARITY_TEST_PASSED = True
print(parity_df.to_string(index=False))
print(f"PARITY TEST PASSED: all max absolute probability differences < {PARITY_TOLERANCE} (TTA disabled).")


Inference threshold = 0.8500 (sidecar+checkpoint)
Preprocessing loaded from checkpoint
dB→gray lo/hi used: -30.0 0.0
PRE/POST compatibility PASSED: width/height/CRS/transform/resolution/bounds match; VV is available in both.
NoData fill gray: PRE=0.0, POST=0.0 (zero)


Inference rows:   0%|          | 0/1 [00:00<?, ?it/s]

Flood area: 0.3570 km² (5.447% of valid area)
Inference threshold = 0.8500 (sidecar+checkpoint)
Preprocessing loaded from checkpoint
dB→gray lo/hi used: -30.0 0.0
PRE/POST compatibility PASSED: width/height/CRS/transform/resolution/bounds match; VV is available in both.
NoData fill gray: PRE=0.0, POST=0.0 (zero)


Inference rows:   0%|          | 0/1 [00:00<?, ?it/s]

Flood area: 0.4477 km² (6.831% of valid area)
Inference threshold = 0.8500 (sidecar+checkpoint)
Preprocessing loaded from checkpoint
dB→gray lo/hi used: -30.0 0.0
PRE/POST compatibility PASSED: width/height/CRS/transform/resolution/bounds match; VV is available in both.
NoData fill gray: PRE=0.0, POST=0.0 (zero)


Inference rows:   0%|          | 0/1 [00:00<?, ?it/s]

Flood area: 2.9340 km² (44.769% of valid area)
                key  max_abs_probability_difference
test/image_4831.png                    5.960464e-08
test/image_4832.png                    5.960464e-08
test/image_4833.png                    5.960464e-08
PARITY TEST PASSED: all max absolute probability differences < 0.01 (TTA disabled).


In [56]:
# Cell 35C — Gaibandha radiometric sensitivity test (diagnostic only; never used to tune the chosen mapping)
# Set ROI_WINDOW=(col_off,row_off,width,height) in Cell 4. Median-fill diagnostics use the selected ROI to keep this test cheap.

if ROI_WINDOW is None:
    print("ROI_WINDOW=None — radiometric sensitivity test skipped. Set a user-chosen ROI in Cell 4 to run it.")
elif not (os.path.exists(EXTERNAL_PRE_TIF) and os.path.exists(EXTERNAL_POST_TIF)):
    print("External GeoTIFFs unavailable — sensitivity test skipped.")
else:
    roi = Window(*map(int, ROI_WINDOW))
    sens_dir = os.path.join(DIRS["inference_outputs"], "radiometric_sensitivity")
    os.makedirs(sens_dir, exist_ok=True)
    sens_rows = []; sens_masks = {}
    sens_model, sens_ckpt = load_model_from_checkpoint(BEST_CKPT_PATH, DEVICE)
    locked_threshold, _ = _resolve_locked_threshold(sens_ckpt, BEST_CKPT_PATH)

    for lo, hi in SENSITIVITY_DB_RANGES:
        for fill_mode in SENSITIVITY_NODATA_FILLS:
            pp = S1GFloodsPreprocessor(lo, hi, PNG_QUANTIZE, sens_ckpt["config"]["IN_CHANNELS"], sens_ckpt["config"]["INPUT_MODE"])
            tag = f"lo{lo:g}_hi{hi:g}_{fill_mode}".replace("-", "m").replace(".", "p")
            out_dir = os.path.join(sens_dir, tag)
            p_path, m_path, v_path, meta = predict_geotiff(
                EXTERNAL_PRE_TIF, EXTERNAL_POST_TIF, BEST_CKPT_PATH, out_dir,
                tile_size=INFERENCE_TILE_SIZE, overlap=INFERENCE_OVERLAP, device=DEVICE,
                tta=False, nodata_fill=fill_mode, calibration_mode="none",
                preprocessor_override=pp, roi_window=roi, diagnostic_fill_from_roi=True,
            )
            with rasterio.open(m_path) as sm, rasterio.open(v_path) as sv:
                mask = sm.read(1); valid = sv.read(1).astype(bool)
                a = area_statistics(mask, valid, sm.transform, sm.crs)
            sens_masks[tag] = (mask == 1, valid)
            sens_rows.append({"tag": tag, "lo": lo, "hi": hi, "nodata_fill": fill_mode,
                              "flood_area_km2": a["flood_area_km2"],
                              "flood_percent_of_valid_area": a["flood_percent_of_valid_area"],
                              "threshold": locked_threshold})

    sens_df = pd.DataFrame(sens_rows)
    sens_df.to_csv(os.path.join(sens_dir, "radiometric_sensitivity_area.csv"), index=False)
    pair_rows = []
    tags = list(sens_masks)
    for i in range(len(tags)):
        for j in range(i+1, len(tags)):
            a, va = sens_masks[tags[i]]; b, vb = sens_masks[tags[j]]
            valid = va & vb; union = ((a | b) & valid).sum(); inter = ((a & b) & valid).sum()
            iou = float(inter / union) if union else 1.0
            pair_rows.append({"setting_a": tags[i], "setting_b": tags[j], "pairwise_mask_iou": iou})
    pairwise_df = pd.DataFrame(pair_rows)
    pairwise_df.to_csv(os.path.join(sens_dir, "radiometric_sensitivity_pairwise_iou.csv"), index=False)
    print("DIAGNOSTIC ONLY — never choose lo/hi from these results to improve external appearance/performance.")
    print(sens_df.to_string(index=False))
    print(pairwise_df.to_string(index=False))


ROI_WINDOW=None — radiometric sensitivity test skipped. Set a user-chosen ROI in Cell 4 to run it.


In [57]:
# Cell 35D — Export external inference GeoTIFF outputs as EPS
# EPS is visualization-only; GeoTIFFs remain the georeferenced source of record.

from matplotlib.colors import ListedColormap

def geotiff_to_eps(tif_path, eps_path, mode="probability"):
    with rasterio.open(tif_path) as src:
        arr = src.read(1, masked=True)
    height, width = arr.shape
    fig, ax = plt.subplots(figsize=(10, 10 * height / width))
    if mode == "probability":
        ax.imshow(arr, cmap="Blues", vmin=0, vmax=1, interpolation="nearest")
    elif mode == "binary":
        ax.imshow(arr, cmap=ListedColormap(["white", "blue"]), vmin=0, vmax=1, interpolation="nearest")
    elif mode == "validity":
        ax.imshow(arr, cmap=ListedColormap(["black", "white"]), vmin=0, vmax=1, interpolation="nearest")
    else:
        raise ValueError("mode must be 'probability', 'binary', or 'validity'")
    ax.axis("off"); fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
    fig.savefig(eps_path, format="eps", bbox_inches="tight", pad_inches=0); plt.close(fig)
    print("Saved:", eps_path)

if ext_out_dir is not None and all(p is not None and os.path.exists(p) for p in [prob_path, mask_path, validity_path]):
    eps_dir = os.path.join(ext_out_dir, "eps"); os.makedirs(eps_dir, exist_ok=True)
    geotiff_to_eps(prob_path, os.path.join(eps_dir, "flood_probability.eps"), "probability")
    geotiff_to_eps(mask_path, os.path.join(eps_dir, "flood_binary_mask.eps"), "binary")
    geotiff_to_eps(validity_path, os.path.join(eps_dir, "validity_mask.eps"), "validity")
    print("EPS export complete:", eps_dir)
else:
    print("External inference outputs are unavailable — EPS export skipped.")


Saved: /kaggle/working/samcd_s1gfloods/inference_outputs/gaibandha/eps/flood_probability.eps
Saved: /kaggle/working/samcd_s1gfloods/inference_outputs/gaibandha/eps/flood_binary_mask.eps
Saved: /kaggle/working/samcd_s1gfloods/inference_outputs/gaibandha/eps/validity_mask.eps
EPS export complete: /kaggle/working/samcd_s1gfloods/inference_outputs/gaibandha/eps


In [58]:
# Cell 36 — Domain-shift report + geospatially correct external area summary
# Domain-shift diagnostics use the RAW external VV values with the checkpoint mapping (no optional two-point calibration),
# so calibration cannot hide the shift being measured.


def _hist_quantile(hist: np.ndarray, q: float) -> float:
    if hist.sum() == 0: return np.nan
    return float(np.searchsorted(np.cumsum(hist), q * hist.sum()))


def gray_hist_stats(hist: np.ndarray) -> Dict[str, float]:
    total = float(hist.sum())
    values = np.arange(256, dtype=np.float64)
    if total == 0:
        return {k: np.nan for k in ["mean","std","p0.5","p1","p5","p25","p50","p75","p95","p99","p99.5","pct_at_0","pct_at_255","pct_below_60","pct_below_99"]}
    mean = float(np.sum(values * hist) / total)
    std = float(np.sqrt(np.sum(((values-mean)**2) * hist) / total))
    out = {"mean": mean, "std": std,
           "pct_at_0": float(100*hist[0]/total), "pct_at_255": float(100*hist[255]/total),
           "pct_below_60": float(100*hist[:60].sum()/total), "pct_below_99": float(100*hist[:99].sum()/total)}
    for q in [0.005,0.01,0.05,0.25,0.50,0.75,0.95,0.99,0.995]:
        out[f"p{100*q:g}"] = _hist_quantile(hist, q)
    return out

if os.path.exists(EXTERNAL_PRE_TIF) and os.path.exists(EXTERNAL_POST_TIF):
    domain_ckpt = load_checkpoint(BEST_CKPT_PATH, "cpu")
    domain_preproc = S1GFloodsPreprocessor.from_dict(domain_ckpt["preprocessing"])
    compat = check_pre_post_compatibility(EXTERNAL_PRE_TIF, EXTERNAL_POST_TIF)
    with rasterio.open(EXTERNAL_PRE_TIF) as pre_src, rasterio.open(EXTERNAL_POST_TIF) as post_src:
        ext_pre_hist = gray_histogram_from_raster(pre_src, compat["pre_vv_index"], domain_preproc, calibration_mode="none")
        ext_post_hist = gray_histogram_from_raster(post_src, compat["post_vv_index"], domain_preproc, calibration_mode="none")

        # Figure window: use ROI_WINDOW if provided; otherwise an explicit centered window capped by config.
        if ROI_WINDOW is not None:
            fig_win = Window(*map(int, ROI_WINDOW))
        else:
            fw = min(DOMAIN_FIGURE_MAX_SIZE, pre_src.width); fh = min(DOMAIN_FIGURE_MAX_SIZE, pre_src.height)
            fig_win = Window((pre_src.width-fw)//2, (pre_src.height-fh)//2, fw, fh)
        pre_db_fig = pre_src.read(compat["pre_vv_index"], window=fig_win).astype(np.float32)
        post_db_fig = post_src.read(compat["post_vv_index"], window=fig_win).astype(np.float32)
        diff_db_fig = post_db_fig - pre_db_fig

    train_A_hist = np.asarray(TRAIN_GRAY_HISTS["A"], dtype=np.int64)
    train_B_hist = np.asarray(TRAIN_GRAY_HISTS["B"], dtype=np.int64)
    hist_map = {"S1GFloods_train_A": train_A_hist, "S1GFloods_train_B": train_B_hist,
                "Gaibandha_PRE": ext_pre_hist, "Gaibandha_POST": ext_post_hist}
    rows = []
    for name, hist in hist_map.items():
        row = {"source": name, **gray_hist_stats(hist)}
        if name == "Gaibandha_PRE":
            row["wasserstein_vs_matching_train"] = float(wasserstein_distance(np.arange(256), np.arange(256),
                                                                               u_weights=ext_pre_hist, v_weights=train_A_hist))
        elif name == "Gaibandha_POST":
            row["wasserstein_vs_matching_train"] = float(wasserstein_distance(np.arange(256), np.arange(256),
                                                                               u_weights=ext_post_hist, v_weights=train_B_hist))
        else:
            row["wasserstein_vs_matching_train"] = 0.0
        rows.append(row)
    domain_shift_df = pd.DataFrame(rows)
    domain_shift_df.to_csv(os.path.join(DIRS["inference_outputs"], "domain_shift_report.csv"), index=False)

    x = np.arange(256)
    for log_scale, suffix in [(False, "linear"), (True, "log")]:
        fig, ax = plt.subplots(figsize=(10, 6))
        for name, hist in hist_map.items():
            density = hist / max(hist.sum(), 1)
            ax.plot(x, density, label=name)
        if log_scale: ax.set_yscale("log")
        ax.set_xlabel("Mapped VV gray (0..255)"); ax.set_ylabel("Density")
        ax.set_title(f"S1GFloods vs Gaibandha gray-domain distributions ({suffix} scale)")
        ax.legend(); fig.savefig(os.path.join(DIRS["inference_outputs"], f"domain_histogram_{suffix}.png"), dpi=200, bbox_inches="tight"); plt.close(fig)

    finite_pair = np.isfinite(pre_db_fig) & np.isfinite(post_db_fig)
    if finite_pair.any():
        both = np.concatenate([pre_db_fig[finite_pair], post_db_fig[finite_pair]])
        vmin, vmax = np.percentile(both, [2, 98])
        dlim = float(np.percentile(np.abs(diff_db_fig[finite_pair]), 98))
    else:
        vmin, vmax, dlim = -30, 0, 10
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(pre_db_fig, cmap="gray", vmin=vmin, vmax=vmax); axes[0].set_title("Gaibandha PRE VV (dB)")
    axes[1].imshow(post_db_fig, cmap="gray", vmin=vmin, vmax=vmax); axes[1].set_title("Gaibandha POST VV (dB)")
    axes[2].imshow(diff_db_fig, cmap="RdBu", vmin=-max(dlim,1e-3), vmax=max(dlim,1e-3)); axes[2].set_title("POST - PRE VV (dB)")
    for ax in axes: ax.axis("off")
    fig.suptitle(f"Domain-shift inspection window {tuple(map(int,[fig_win.col_off,fig_win.row_off,fig_win.width,fig_win.height]))}")
    fig.tight_layout(); fig.savefig(os.path.join(DIRS["inference_outputs"], "gaibandha_pre_post_difference.png"), dpi=200, bbox_inches="tight"); plt.close(fig)

    train_dark = gray_hist_stats(train_A_hist)["pct_below_60"]
    ext_dark = gray_hist_stats(ext_pre_hist)["pct_below_60"]
    print(domain_shift_df.to_string(index=False))
    if train_dark > 0 and ext_dark < 0.5 * train_dark:
        print(f"WARNING: Gaibandha PRE dark-tail fraction (<gray 60) is far below training A: {ext_dark:.2f}% vs {train_dark:.2f}%. "
              "Flood recall may be under-estimated under this radiometric shift.")
    print("UNVERIFIED mapping used for this report:", (domain_preproc.lo, domain_preproc.hi))

    expected_mask = mask_path if mask_path is not None and os.path.exists(mask_path) else os.path.join(DIRS["inference_outputs"], "gaibandha", "flood_binary_mask.tif")
    expected_valid = validity_path if validity_path is not None and os.path.exists(validity_path) else os.path.join(DIRS["inference_outputs"], "gaibandha", "validity_mask.tif")
    if os.path.exists(expected_mask) and os.path.exists(expected_valid):
        with rasterio.open(expected_mask) as src_m, rasterio.open(expected_valid) as src_v:
            mask_arr = src_m.read(1); valid_arr = src_v.read(1).astype(bool)
            area_summary_dict = area_statistics(mask_arr, valid_arr, src_m.transform, src_m.crs)
        area_summary = pd.DataFrame([area_summary_dict])
        area_summary.to_csv(os.path.join(DIRS["inference_outputs"], "external_area_summary.csv"), index=False)
        print(area_summary.to_string(index=False))
    else:
        print("External prediction outputs not found; domain-shift report saved, area summary skipped.")
else:
    print("External GeoTIFFs unavailable — domain-shift report and area calculation skipped.")


PRE/POST compatibility PASSED: width/height/CRS/transform/resolution/bounds match; VV is available in both.
           source       mean       std  pct_at_0  pct_at_255  pct_below_60  pct_below_99  p0.5   p1    p5   p25   p50   p75   p95   p99  p99.5  wasserstein_vs_matching_train
S1GFloods_train_A 130.579816 62.055686  6.602232    2.643188     14.450976     26.170156   0.0  0.0   0.0  96.0 136.0 176.0 224.0 255.0  255.0                       0.000000
S1GFloods_train_B 116.780180 78.129187  9.278441    1.824886     30.760816     44.827569   0.0  0.0   0.0  47.0 119.0 189.0 230.0 255.0  255.0                       0.000000
    Gaibandha_PRE 176.849380 28.308310  0.000000    0.109992      0.002940      2.513693  84.0 89.0 113.0 169.0 181.0 193.0 214.0 232.0  239.0                      48.467536
   Gaibandha_POST 159.585611 43.789873  0.002139    0.451401      0.747596     13.715751  55.0 63.0  82.0 122.0 173.0 191.0 216.0 240.0  253.0                      46.439331
UNVERIFIED mapping use

In [59]:
# Cell 37 — Experiment manifest + environment export

def get_pip_freeze():
    try:
        out = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, timeout=60)
        return out.stdout
    except Exception as e:
        return f"pip freeze failed: {e}"

env_info = {
    "python_version": platform.python_version(), "torch_version": torch.__version__,
    "cuda_version": torch.version.cuda, "cudnn_version": torch.backends.cudnn.version(),
    "gpu_count": N_GPUS, "gpu_names": [torch.cuda.get_device_properties(i).name for i in range(N_GPUS)] if N_GPUS else [],
    "timestamp_utc": pd.Timestamp.utcnow().isoformat(), "seed": SEED,
}
with open(os.path.join(DIRS["environment"], "environment_info.json"), "w") as f: json.dump(env_info, f, indent=2)
with open(os.path.join(DIRS["environment"], "pip_freeze.txt"), "w") as f: f.write(get_pip_freeze())

experiment_manifest = {
    "dataset": {
        "name": "S1GFloods", "data_root": DATA_ROOT, "split_source": split_source,
        "counts": {k: len(v) for k, v in SPLITS.items()},
        "label_semantics": "GT=255 / GT>127 = newly inundated flood; permanent water negative",
        "split_independence_caveat": "filenames have no event/region ID; event-level independence cannot be verified from filenames",
    },
    "model": MODEL_NAME, "input_mode": INPUT_MODE, "in_channels_per_date": IN_CHANNELS,
    "model_reference": {
        "name": "SAM-CD", "paper": "Ding, Zhu, Peng, Tang, Yang, Bruzzone, IEEE TGRS 62 (2024), doi:10.1109/TGRS.2024.3368168",
        "code": "https://github.com/ggsDing/SAM-CD", "encoder": "FastSAM-x (frozen; YOLOv8x-seg layers 0-21)",
        "encoder_weights": SAMCD_WEIGHTS, "use_tsl_aux_loss": SAMCD_USE_TSL, "tsl_temperature": SAMCD_TSL_TEMPERATURE,
        "total_params": int(total_params), "trainable_params": int(trainable_params),
        "max_train_hours_guard": MAX_TRAIN_HOURS,
        "trained_epochs_note": "if epochs_run < epochs_configured, training stopped by early stopping or MAX_TRAIN_HOURS",
    },
    "loss": LOSS_NAME, "optimizer": "AdamW", "scheduler": SCHEDULER_NAME,
    "seed": SEED, "run_seeds": RUN_SEEDS, "epochs_configured": EPOCHS,
    "epochs_run": int(hist_df["epoch"].max()) + 1 if len(hist_df) else 0,
    "best_epoch": int(BEST_CKPT["epoch"]), "learning_rate": LEARNING_RATE,
    "batch_size_per_gpu": BATCH_SIZE_PER_GPU, "accum_steps": ACCUM_STEPS,
    "normalization": PREPROCESSOR.to_dict(),
    "unverified_assumptions": {"png_db_range": [PNG_DB_LO, PNG_DB_HI], "external_input_is_composite": EXTERNAL_INPUT_IS_COMPOSITE},
    "augmentations": {"spatial": USE_AUGMENTATION, "radiometric": USE_SAR_RADIOMETRIC_AUG,
                      "dark_tail": AUG_DARK_TAIL, "gain_offset": AUG_GAIN_OFFSET, "date_offset": AUG_DATE_OFFSET, "blur": AUG_BLUR},
    "threshold": THRESHOLD, "inference_tta": INFERENCE_TTA,
    "test_metrics": {k: (float(v) if isinstance(v, (int,float,np.floating,np.integer)) else v) for k,v in global_metrics.items()},
    "checkpoint": BEST_CKPT_PATH, "software_versions": env_info, "gpu": env_info["gpu_names"],
    "training_strategy": "DDP" if EFFECTIVE_DDP else ("DataParallel" if EFFECTIVE_DP else "single-GPU"),
    "runtime": complexity_df.to_dict("records")[0] if len(complexity_df) else {},
    "external_outputs_validated_claim": False,
    "validation_note": "Do not call external GeoTIFF outputs validated unless the GeoTIFF/PNG parity test passes and any supplied reference-mask evaluation also passes.",
}
with open(os.path.join(OUTPUT_ROOT, "experiment_manifest.json"), "w") as f:
    json.dump(experiment_manifest, f, indent=2, default=str)
print("Saved experiment_manifest.json and environment exports.")

Saved experiment_manifest.json and environment exports.


In [60]:
# Cell 38 — Zip all outputs for download from Kaggle

zip_path = "/kaggle/working/samcd_s1gfloods_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(OUTPUT_ROOT):
        for file in files:
            full_path = os.path.join(root, file)
            zf.write(full_path, os.path.relpath(full_path, OUTPUT_ROOT))
size_mb = os.path.getsize(zip_path) / (1024 ** 2)
print(f"Zipped all outputs to {zip_path} ({size_mb:.1f} MB).")
for d in SUBDIRS:
    p = os.path.join(OUTPUT_ROOT, d); n = sum(len(files) for _, _, files in os.walk(p))
    print(f"  {d:<20s}: {n} files")

Zipped all outputs to /kaggle/working/samcd_s1gfloods_outputs.zip (449.8 MB).
  checkpoints         : 3 files
  logs                : 1 files
  metrics             : 14 files
  figures             : 12 files
  predictions         : 0 files
  qualitative         : 18 files
  configs             : 2 files
  inference_outputs   : 35 files
  environment         : 2 files
  split_audit         : 6 files


## Summary / How to use this notebook

**1. Environment.** Kaggle GPU accelerator + Internet ON. Cell 2B installs `ultralytics` (only needed to un-pickle the encoder) and Cell 2C downloads `FastSAM-x.pt` (~145 MB) to `/kaggle/working/pretrained_weight/`. If Internet is unavailable, attach a dataset containing `FastSAM-x.pt`.

**2. Data.** `DATA_ROOT` is the same flat `A/B/Label` S1GFloods folder as the Siamese-UNet notebook, with the same deterministic natural-filename split (4300/530/530). Nothing about the data pipeline was changed, so both models see the same images, preprocessing and augmentation.

**3. Model.** `MODEL_NAME="sam_cd"` — SAM-CD (Ding et al., IEEE TGRS 2024) with a frozen FastSAM-x encoder, convolutional adaptors, per-date decoders and a spatial-attention-gated change head. The single-channel VV tensor is replicated to 3 channels *inside* the model; `INPUT_MODE` remains `"1ch"`.

**4. Training.** All hyper-parameters are those of the Siamese-UNet notebook (200 epochs, AdamW 1e-4, wd 1e-4, 5-epoch warm-up + cosine, `bce_dice`, micro-batch 4 × 4 accumulation, AMP, early-stopping patience 50, seed 42, single GPU). `SAMCD_USE_TSL=False` keeps the loss identical to the baseline; set it to `True` to also use the paper's task-agnostic semantic loss (report it as a separate variant, since the loss then differs from the baseline).

**5. Frozen encoder.** Only adaptors/decoders/head are trained (see `trainable_params` in `metrics/complexity_runtime.csv` and `paper_results_table.csv`). Report both total and trainable parameters.

**6. Time guard.** `MAX_TRAIN_HOURS=10.5` stops training cleanly (and still evaluates/exports) so a 12 h Kaggle session does not lose everything. If it triggers, the cosine schedule did not complete; state this in the paper or re-run on a longer budget.

**7. Comparison.** Set `BASELINE_RESULTS_DIR` to the Siamese-UNet output folder to get `benchmark_comparison_*.csv` (global metrics, paired image-level bootstrap CI and Wilcoxon test, complexity).

**8. Everything else** (threshold search, test evaluation, bootstrap CIs, qualitative/EPS figures, GeoTIFF inference, parity test, sensitivity test, domain-shift report, manifest, zip) is unchanged from the Siamese-UNet notebook, including its UNVERIFIED-assumption caveats and reporting language.
